In [1]:
import sys
print("python exe:", sys.executable)   # full path to the interpreter
print("python ver:", sys.version)

# try torch only in the working notebook
try:
    import torch
    print("torch      :", torch.__version__, torch.__file__)
except ModuleNotFoundError as e:
    print("torch not importable:", e)



python exe: c:\Users\aneek\anaconda3\envs\tf_gpu_env\python.exe
python ver: 3.9.23 | packaged by conda-forge | (main, Jun  4 2025, 17:49:16) [MSC v.1929 64 bit (AMD64)]


c:\Users\aneek\anaconda3\envs\tf_gpu_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch      : 1.12.1+cu113 c:\Users\aneek\anaconda3\envs\tf_gpu_env\lib\site-packages\torch\__init__.py


In [2]:
import tensorflow as tf
print(tf.__version__)  # This should print the version of TensorFlow
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

print("CUDA version:", tf.sysconfig.get_build_info()["cuda_version"])
print("cuDNN version:", tf.sysconfig.get_build_info()["cudnn_version"])

from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())

2.10.0
Num GPUs Available:  1
CUDA version: 64_112
cuDNN version: 64_8
[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 7885901842333210387
xla_global_id: -1
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 5713690624
locality {
  bus_id: 1
  links {
  }
}
incarnation: 12050353341665036177
physical_device_desc: "device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9"
xla_global_id: 416903419
]


In [3]:
import time
import tensorflow as tf
import psutil

class PowerMonitor:
    def __init__(self):
        self.gpu_available = tf.config.list_physical_devices('GPU')
        
        # Hardware power specifications (adjust these values for your system)
        self.cpu_tdp = 65    # Typical TDP for desktop CPUs in watts
        self.gpu_tdp = 250   # Typical TDP for desktop GPUs in watts
        
    def get_stats(self):
        """Get system stats with power estimation"""
        stats = {
            'timestamp': time.time(),
            'cpu_%': psutil.cpu_percent(interval=0.1),
            'ram_mb': psutil.virtual_memory().used / (1024**2),
            'gpu_mem_mb': 0,
            'power_w': self.cpu_tdp * (psutil.cpu_percent()/100) * 0.85  # Base CPU power
        }
        
        if self.gpu_available:
            try:
                # TensorFlow GPU memory monitoring
                mem_info = tf.config.experimental.get_memory_info('GPU:0')
                stats.update({
                    'gpu_mem_mb': mem_info['current'] / (1024**2),
                    'power_w': self.cpu_tdp * (psutil.cpu_percent()/100) * 0.85 + 
                              self.gpu_tdp * 0.5 * 0.75  # Add GPU power estimate
                })
            except:
                pass
                
        return stats

# Initialize monitor
monitor = PowerMonitor()

In [4]:
import time
import torch
import psutil
import os

class PowerMonitor1:
    def __init__(self):
        self.gpu_available = torch.cuda.is_available()
        self.process = psutil.Process(os.getpid())  # Track current process
        
        # Hardware power specifications
        self.cpu_tdp = 65
        self.gpu_tdp = 250
        
    def get_stats(self):
        """Get process-specific stats with power estimation"""
        process_memory = self.process.memory_info()
        
        stats = {
            'timestamp': time.time(),
            'cpu_%': psutil.cpu_percent(interval=0.1),
            'process_ram_mb': process_memory.rss / (1024**2),  # Only this process's RAM
            'gpu_mem_mb': 0,
            'power_w': self.cpu_tdp * (psutil.cpu_percent()/100) * 0.85
        }
        
        if self.gpu_available:
            try:
                gpu_memory_allocated = torch.cuda.memory_allocated()
                stats.update({
                    'gpu_mem_mb': gpu_memory_allocated / (1024**2),
                    'power_w': self.cpu_tdp * (psutil.cpu_percent()/100) * 0.85 + 
                              self.gpu_tdp * 0.5 * 0.75
                })
            except Exception as e:
                print(f"Error retrieving GPU memory: {e}")
                
        return stats

# Initialize monitor1
monitor1 = PowerMonitor1()

# Model

In [5]:
# ============================================================
# MAXVIT-BASE: IMPORTS AND CONFIGURATION
# ============================================================

import random
import numpy as np
import torch
import torch.optim as optim
import timm

from torch.utils.data import Dataset, DataLoader
from PIL import Image

from torchvision import transforms
from torchvision.transforms import InterpolationMode


# ============================================================
# CONFIGURATION
# ============================================================

MODEL_NAME = "maxvit_base_tf_224.in1k"

INPUT_SIZE = 160
NUM_CLASSES = 2
BATCH_SIZE = 16
EPOCHS = 10
LEARNING_RATE = 1e-4
RANDOM_SEED = 42


device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)


# ============================================================
# REPRODUCIBILITY
# ============================================================

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(
        RANDOM_SEED
    )

# More compatible with convolution-heavy MaxViT models
torch.backends.cudnn.enabled = True
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True


# ============================================================
# LOAD IMAGENET-1K PRETRAINED MAXVIT-BASE
# ============================================================

try:
    model = timm.create_model(
        MODEL_NAME,
        pretrained=True,
        num_classes=NUM_CLASSES,

        # Reconfigure MaxViT partitioning for 160 × 160 input
        img_size=INPUT_SIZE
    )

except (RuntimeError, KeyError):
    # Compatibility fallback for older timm versions
    MODEL_NAME = "maxvit_base_tf_224"

    model = timm.create_model(
        MODEL_NAME,
        pretrained=True,
        num_classes=NUM_CLASSES,
        img_size=INPUT_SIZE
    )


# Full-model fine-tuning
for parameter in model.parameters():
    parameter.requires_grad = True


model = model.to(device)


# ============================================================
# MODEL INFORMATION
# ============================================================

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)


print("Model:", MODEL_NAME)
print("Output classes:", NUM_CLASSES)
print("Trainable parameters:", trainable_parameters)
print("Total parameters:", total_parameters)

print("\nPretrained configuration:")
print(model.pretrained_cfg)
# ============================================================
# MAXVIT PREPROCESSING
# ============================================================

maxvit_transform = transforms.Compose([
    transforms.Resize(
        (INPUT_SIZE, INPUT_SIZE),
        interpolation=InterpolationMode.BICUBIC,
        antialias=True
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

Device: cuda


c:\Users\aneek\anaconda3\envs\tf_gpu_env\lib\site-packages\timm\layers\interpolate.py:47: UserWarning: torch.searchsorted(): input value tensor is non-contiguous, this will lower the performance due to extra data copy when converting non-contiguous tensor to contiguous, please use contiguous input value tensor if possible. This message will only appear once per program. (Triggered internally at  C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen/native/BucketizationUtils.h:35.)
  idx_right = torch.bucketize(x, p)


Model: maxvit_base_tf_224.in1k
Output classes: 2
Trainable parameters: 118654838
Total parameters: 118654838

Pretrained configuration:
{'url': '', 'hf_hub_id': 'timm/maxvit_base_tf_224.in1k', 'architecture': 'maxvit_base_tf_224', 'tag': 'in1k', 'custom_load': False, 'input_size': (3, 224, 224), 'fixed_input_size': True, 'interpolation': 'bicubic', 'crop_pct': 0.95, 'crop_mode': 'center', 'mean': (0.485, 0.456, 0.406), 'std': (0.229, 0.224, 0.225), 'num_classes': 1000, 'pool_size': (7, 7), 'first_conv': 'stem.conv1', 'classifier': 'head.fc'}


In [6]:
# ============================================================
# MAXVIT DATASET
# ============================================================

class DeepfakeMaxViTDataset(Dataset):
    """
    Dataset for MaxViT deepfake classification.

    Images:
        OpenCV BGR NumPy arrays, PIL images,
        or image file paths.

    Labels:
        0 = real
        1 = fake
    """

    def __init__(
        self,
        images,
        labels,
        transform=None
    ):
        self.images = images

        self.labels = np.asarray(
            labels,
            dtype=np.int64
        )

        self.transform = transform

        if len(self.images) != len(self.labels):
            raise ValueError(
                "The numbers of images and labels do not match."
            )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        image = self.images[index]
        label = int(self.labels[index])

        # ----------------------------------------------------
        # NUMPY IMAGE
        # ----------------------------------------------------

        if isinstance(image, np.ndarray):

            if (
                image.ndim != 3
                or image.shape[-1] != 3
            ):
                raise ValueError(
                    f"Invalid image shape at index "
                    f"{index}: {image.shape}"
                )

            # OpenCV BGR -> RGB
            image = image[..., ::-1]

            image = np.ascontiguousarray(
                image,
                dtype=np.uint8
            )

            image = Image.fromarray(
                image,
                mode="RGB"
            )

        # ----------------------------------------------------
        # IMAGE PATH
        # ----------------------------------------------------

        elif isinstance(image, str):

            with Image.open(image) as opened_image:
                image = opened_image.convert("RGB")

        # ----------------------------------------------------
        # PIL IMAGE
        # ----------------------------------------------------

        elif isinstance(image, Image.Image):

            image = image.convert("RGB")

        else:
            raise TypeError(
                f"Unsupported image type at index "
                f"{index}: {type(image)}"
            )

        if self.transform is not None:
            image = self.transform(image)

        label = torch.tensor(
            label,
            dtype=torch.long
        )

        return image, label

In [7]:
# ============================================================
# MAXVIT VALIDATION FUNCTION
# ============================================================

def evaluate_epoch(
    model,
    data_loader,
    criterion,
    device
):
    model.eval()

    total_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    with torch.inference_mode():

        for images, labels in data_loader:

            images = images.to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

            labels = labels.to(
                device,
                dtype=torch.long,
                non_blocking=True
            )

            # timm MaxViT returns logits directly
            logits = model(images)

            loss = criterion(
                logits,
                labels
            )

            predictions = torch.argmax(
                logits,
                dim=1
            )

            batch_size = labels.size(0)

            total_loss += (
                loss.item() * batch_size
            )

            correct_predictions += (
                predictions == labels
            ).sum().item()

            total_samples += batch_size

    if total_samples == 0:
        raise RuntimeError(
            "The validation DataLoader is empty."
        )

    val_loss = (
        total_loss / total_samples
    )

    val_accuracy = (
        correct_predictions / total_samples
    )

    return val_loss, val_accuracy
# ============================================================
# MAXVIT TRAINING FUNCTION
# ============================================================

def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    device,
    epochs=10
):
    history = {
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": []
    }

    for epoch in range(epochs):

        model.train()

        total_train_loss = 0.0
        correct_train_predictions = 0
        total_train_samples = 0

        for images, labels in train_loader:

            images = images.to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

            labels = labels.to(
                device,
                dtype=torch.long,
                non_blocking=True
            )

            optimizer.zero_grad()

            # timm MaxViT forward pass
            logits = model(images)

            loss = criterion(
                logits,
                labels
            )

            loss.backward()
            optimizer.step()

            predictions = torch.argmax(
                logits,
                dim=1
            )

            batch_size = labels.size(0)

            total_train_loss += (
                loss.item() * batch_size
            )

            correct_train_predictions += (
                predictions == labels
            ).sum().item()

            total_train_samples += batch_size

        if total_train_samples == 0:
            raise RuntimeError(
                "The training DataLoader is empty."
            )

        train_loss = (
            total_train_loss
            / total_train_samples
        )

        train_accuracy = (
            correct_train_predictions
            / total_train_samples
        )

        val_loss, val_accuracy = evaluate_epoch(
            model=model,
            data_loader=val_loader,
            criterion=criterion,
            device=device
        )

        history["train_loss"].append(
            train_loss
        )

        history["train_accuracy"].append(
            train_accuracy
        )

        history["val_loss"].append(
            val_loss
        )

        history["val_accuracy"].append(
            val_accuracy
        )

        print(
            f"Epoch {epoch + 1:02d}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Accuracy: {train_accuracy:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Accuracy: {val_accuracy:.4f}"
        )

    return history
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=1e-4)
# ============================================================
# COMPLETE maxvit EVALUATION FUNCTION
# ============================================================

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve,
    auc
)


def evaluate_complete(
    model,
    loader,
    criterion,
    device
):
    model.eval()

    total_loss = 0.0
    total_samples = 0

    all_labels = []
    all_predictions = []
    all_fake_probabilities = []

    with torch.inference_mode():

        for images, labels in loader:

            images = images.to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

            labels = labels.to(
                device,
                dtype=torch.long,
                non_blocking=True
            )

            # timm MaxViT returns logits directly
            logits = model(images)

            loss = criterion(
                logits,
                labels
            )

            probabilities = torch.softmax(
                logits,
                dim=1
            )

            predictions = torch.argmax(
                logits,
                dim=1
            )

            fake_probabilities = probabilities[:, 1]

            batch_size = labels.size(0)

            total_loss += (
                loss.item() * batch_size
            )

            total_samples += batch_size

            all_labels.extend(
                labels.detach().cpu().numpy()
            )

            all_predictions.extend(
                predictions.detach().cpu().numpy()
            )

            all_fake_probabilities.extend(
                fake_probabilities
                .detach()
                .cpu()
                .numpy()
            )

    if total_samples == 0:
        raise RuntimeError(
            "The test DataLoader contains no samples."
        )

    y_true = np.asarray(
        all_labels,
        dtype=np.int64
    )

    y_pred = np.asarray(
        all_predictions,
        dtype=np.int64
    )

    y_score = np.asarray(
        all_fake_probabilities,
        dtype=np.float64
    )

    # Keep the remainder of your existing metrics code here.

    # ========================================================
    # CONFUSION MATRIX
    # ========================================================

    confusion = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = confusion.ravel()


    # ========================================================
    # CLASSIFICATION METRICS
    # ========================================================

    average_loss = (
        total_loss / total_samples
    )

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    balanced_accuracy = (
        balanced_accuracy_score(
            y_true,
            y_pred
        )
    )

    precision = precision_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    f1 = f1_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    mcc = matthews_corrcoef(
        y_true,
        y_pred
    )

    false_positive_rate = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else np.nan
    )

    false_negative_rate = (
        fn / (fn + tp)
        if (fn + tp) > 0
        else np.nan
    )


    # ========================================================
    # ROC-AUC, PR-AUC, AP, AND EER
    # ========================================================

    if len(np.unique(y_true)) == 2:

        roc_auc = roc_auc_score(
            y_true,
            y_score
        )

        average_precision = (
            average_precision_score(
                y_true,
                y_score
            )
        )

        pr_precision, pr_recall, _ = (
            precision_recall_curve(
                y_true,
                y_score,
                pos_label=1
            )
        )

        pr_auc = auc(
            pr_recall,
            pr_precision
        )

        roc_fpr, roc_tpr, roc_thresholds = (
            roc_curve(
                y_true,
                y_score,
                pos_label=1
            )
        )

        roc_fnr = 1.0 - roc_tpr

        # Approximate equal-error-rate point
        eer_index = np.nanargmin(
            np.abs(
                roc_fpr - roc_fnr
            )
        )

        eer = (
            roc_fpr[eer_index]
            + roc_fnr[eer_index]
        ) / 2.0

        eer_threshold = (
            roc_thresholds[eer_index]
        )

    else:
        roc_auc = np.nan
        pr_auc = np.nan
        average_precision = np.nan
        eer = np.nan
        eer_threshold = np.nan


    # ========================================================
    # CLASSIFICATION REPORT
    # ========================================================

    report = classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=[
            "real",
            "fake"
        ],
        digits=4,
        zero_division=0
    )


    # ========================================================
    # RESULT DICTIONARY
    # ========================================================

    results = {
        "test_loss": average_loss,
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "precision": precision,
        "recall_sensitivity": recall,
        "specificity": specificity,
        "f1_score": f1,
        "mcc": mcc,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "average_precision": average_precision,
        "eer": eer,
        "eer_threshold": eer_threshold,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp),
        "number_of_test_images": int(total_samples),
        "confusion_matrix": confusion,
        "classification_report": report
    }


    predictions_df = pd.DataFrame({
        "true_label": y_true,
        "predicted_label": y_pred,
        "fake_probability": y_score
    })

    results["predictions"] = predictions_df

    return results

# Wild deepfake

In [20]:
import h5py
import numpy as np

H5_PATH = (
    r"D:\thesis\dataset\WildDeepfake\leakage_free_subset"
    r"\wilddeepfake_sequence_disjoint_face_preprocessed.h5"
)

with h5py.File(H5_PATH, "r") as h5f:
    # Load image arrays
    train_images = h5f["train_images"][:]
    train_labels = h5f["train_labels"][:]

    val_images = h5f["val_images"][:]
    val_labels = h5f["val_labels"][:]

    test_images = h5f["test_images"][:]
    test_labels = h5f["test_labels"][:]

# Verify dataset sizes
print(f"Total train: {len(train_images)} images")
print(f"Total validation: {len(val_images)} images")
print(f"Total test: {len(test_images)} images")

print(f"Train labels: {len(train_labels)}")
print(f"Validation labels: {len(val_labels)}")
print(f"Test labels: {len(test_labels)}")

# Verify shapes and data types
print("\nArray information:")
print(f"Train images: {train_images.shape}, dtype={train_images.dtype}")
print(f"Validation images: {val_images.shape}, dtype={val_images.dtype}")
print(f"Test images: {test_images.shape}, dtype={test_images.dtype}")

print(f"Train labels: {train_labels.shape}, dtype={train_labels.dtype}")
print(f"Validation labels: {val_labels.shape}, dtype={val_labels.dtype}")
print(f"Test labels: {test_labels.shape}, dtype={test_labels.dtype}")

# Verify class distributions
print("\nClass distribution:")
print(
    f"Train: Real={np.sum(train_labels == 0)}, "
    f"Fake={np.sum(train_labels == 1)}"
)
print(
    f"Validation: Real={np.sum(val_labels == 0)}, "
    f"Fake={np.sum(val_labels == 1)}"
)
print(
    f"Test: Real={np.sum(test_labels == 0)}, "
    f"Fake={np.sum(test_labels == 1)}"
)

Total train: 36000 images
Total validation: 6000 images
Total test: 18000 images
Train labels: 36000
Validation labels: 6000
Test labels: 18000

Array information:
Train images: (36000, 160, 160, 3), dtype=uint8
Validation images: (6000, 160, 160, 3), dtype=uint8
Test images: (18000, 160, 160, 3), dtype=uint8
Train labels: (36000,), dtype=uint8
Validation labels: (6000,), dtype=uint8
Test labels: (18000,), dtype=uint8

Class distribution:
Train: Real=9000, Fake=27000
Validation: Real=1500, Fake=4500
Test: Real=4500, Fake=13500


In [11]:
train_dataset = DeepfakeMaxViTDataset(images=train_images,labels=train_labels,transform=maxvit_transform)
val_dataset = DeepfakeMaxViTDataset(images=val_images,labels=val_labels,transform=maxvit_transform)
test_dataset = DeepfakeMaxViTDataset(images=test_images,labels=test_labels,transform=maxvit_transform)

train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())


print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Testing samples:", len(test_dataset))

Training samples: 36000
Validation samples: 6000
Testing samples: 18000


In [12]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    epochs=10
)

C:\Users\aneek\AppData\Local\Temp\ipykernel_30976\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(


Epoch 01/10 | Train Loss: 0.1655 | Train Accuracy: 0.9334 | Val Loss: 0.1296 | Val Accuracy: 0.9488
Epoch 02/10 | Train Loss: 0.0481 | Train Accuracy: 0.9835 | Val Loss: 0.1653 | Val Accuracy: 0.9432
Epoch 03/10 | Train Loss: 0.0286 | Train Accuracy: 0.9902 | Val Loss: 0.1896 | Val Accuracy: 0.9438
Epoch 04/10 | Train Loss: 0.0244 | Train Accuracy: 0.9916 | Val Loss: 0.1754 | Val Accuracy: 0.9348
Epoch 05/10 | Train Loss: 0.0167 | Train Accuracy: 0.9943 | Val Loss: 0.2059 | Val Accuracy: 0.9427
Epoch 06/10 | Train Loss: 0.0153 | Train Accuracy: 0.9951 | Val Loss: 0.2395 | Val Accuracy: 0.9412
Epoch 07/10 | Train Loss: 0.0151 | Train Accuracy: 0.9949 | Val Loss: 0.1659 | Val Accuracy: 0.9457
Epoch 08/10 | Train Loss: 0.0131 | Train Accuracy: 0.9960 | Val Loss: 0.1518 | Val Accuracy: 0.9555
Epoch 09/10 | Train Loss: 0.0117 | Train Accuracy: 0.9963 | Val Loss: 0.1605 | Val Accuracy: 0.9543
Epoch 10/10 | Train Loss: 0.0111 | Train Accuracy: 0.9962 | Val Loss: 0.1849 | Val Accuracy: 0.9535


In [20]:
print("=== DATA LOADING ===")
start = monitor.get_stats()

=== DATA LOADING ===


In [21]:
print("=== DATA LOADING ===")
start = monitor1.get_stats()

=== DATA LOADING ===


In [23]:
test_results = evaluate_complete(
    model=model,
    loader=test_loader,
    criterion=criterion,
    device=device
)

C:\Users\aneek\AppData\Local\Temp\ipykernel_30976\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(


In [ ]:
print("\nMAXVIT-BASE TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if metric in [
        "confusion_matrix",
        "classification_report",
        "predictions"
    ]:
        continue

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )
    else:
        print(
            f"{metric:30s}: {value}"
        )

MAXVIT-BASE TEST RESULTS
test_loss                     : 0.870561
accuracy                      : 0.8118
balanced_accuracy             : 0.7253
precision                     : 0.8576
recall_sensitivity            : 0.8982
specificity                   : 0.5524
f1_score                      : 0.8775
mcc                           : 0.4755
roc_auc                       : 0.8416
pr_auc                        : 0.9371
average_precision             : 0.9371
eer                           : 0.2373
eer_threshold                 : 0.375326
false_positive_rate           : 0.4475
false_negative_rate           : 0.1018
true_negatives                : 2486
false_positives               : 2014
false_negatives               : 1374
true_positives                : 12126
number_of_test_images         : 18000


In [25]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
#print(f"RAM Used: {end['ram_mb'] - start['ram_mb']:.1f} MB")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 18.1%
Time Usage: 437.9 s
GPU Memory Used: 0.0 MB
Power Consumption: 93W


In [26]:
end = monitor1.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 18.6%
Time Usage: 438.3 s
GPU Memory Used: 928.6 MB
Power Consumption: 93W


save the model

In [27]:
# ============================================================
# SAVE TIMM MAXVIT MODEL
# ============================================================

import os
import torch


SAVE_DIR = os.path.join(
    r"D:\thesis\results",
    "maxvit_wild_160"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)


CHECKPOINT_PATH = os.path.join(
    SAVE_DIR,
    "training_checkpoint.pt"
)

WEIGHTS_PATH = os.path.join(
    SAVE_DIR,
    "maxvit_weights.pth"
)


# Save model weights only
torch.save(
    model.state_dict(),
    WEIGHTS_PATH
)


# Save full training checkpoint
torch.save(
    {
        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "history":
            history,

        "model_name":
            MODEL_NAME,

        "input_size":
            INPUT_SIZE,

        "num_classes":
            NUM_CLASSES,

        "epochs":
            EPOCHS,

        "batch_size":
            BATCH_SIZE,

        "learning_rate":
            LEARNING_RATE,

        "random_seed":
            RANDOM_SEED,

        "label_mapping": {
            0: "real",
            1: "fake"
        },

        "normalization_mean": [
            0.485,
            0.456,
            0.406
        ],

        "normalization_std": [
            0.229,
            0.224,
            0.225
        ]
    },

    CHECKPOINT_PATH
)


print("MaxViT model saved successfully.")
print("Weights:", WEIGHTS_PATH)
print("Checkpoint:", CHECKPOINT_PATH)

MaxViT model saved successfully.
Weights: D:\thesis\results\maxvit_wild_160\maxvit_weights.pth
Checkpoint: D:\thesis\results\maxvit_wild_160\training_checkpoint.pt


load the model

In [28]:
# ============================================================
# RESOURCE MONITOR DEFINITION
# RUN THIS BEFORE THE ViT PROFILING CELL
# ============================================================

import os
import time
import threading
import numpy as np
import pandas as pd
import psutil

from pynvml import (
    nvmlInit,
    nvmlShutdown,
    nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetMemoryInfo,
    nvmlDeviceGetUtilizationRates,
    nvmlDeviceGetPowerUsage,
    NVMLError
)


SAMPLING_INTERVAL = 0.1
GPU_INDEX = 0


class ResourceMonitor:
    """
    Continuously samples CPU, RAM, GPU memory,
    GPU utilization, and GPU power.
    """

    def __init__(
        self,
        interval=0.1,
        gpu_index=0
    ):
        self.interval = interval

        self.process = psutil.Process(
            os.getpid()
        )

        self.logical_cpu_count = (
            psutil.cpu_count(logical=True) or 1
        )

        self.stop_event = threading.Event()
        self.samples = []
        self.thread = None

        nvmlInit()

        self.gpu_handle = (
            nvmlDeviceGetHandleByIndex(
                gpu_index
            )
        )

    def _read_gpu_power(self):
        try:
            return (
                nvmlDeviceGetPowerUsage(
                    self.gpu_handle
                ) / 1000.0
            )
        except NVMLError:
            return np.nan

    def _collect_sample(self):
        timestamp = time.perf_counter()

        ram_mb = (
            self.process.memory_info().rss
            / (1024 ** 2)
        )

        process_cpu_raw = (
            self.process.cpu_percent(
                interval=None
            )
        )

        process_cpu_normalized = (
            process_cpu_raw
            / self.logical_cpu_count
        )

        gpu_memory = nvmlDeviceGetMemoryInfo(
            self.gpu_handle
        )

        gpu_memory_used_mb = (
            gpu_memory.used
            / (1024 ** 2)
        )

        gpu_utilization = (
            nvmlDeviceGetUtilizationRates(
                self.gpu_handle
            ).gpu
        )

        gpu_power_w = self._read_gpu_power()

        self.samples.append({
            "timestamp": timestamp,
            "ram_mb": ram_mb,
            "cpu_percent": process_cpu_normalized,
            "gpu_memory_mb": gpu_memory_used_mb,
            "gpu_utilization_percent": gpu_utilization,
            "gpu_power_w": gpu_power_w
        })

    def _sampling_loop(self):
        while not self.stop_event.is_set():
            try:
                self._collect_sample()
            except Exception as error:
                print(
                    "Monitoring warning:",
                    error
                )

            self.stop_event.wait(
                self.interval
            )

    def start(self):
        # Initialize the CPU utilization counter
        self.process.cpu_percent(
            interval=None
        )

        # First observation is the baseline
        self._collect_sample()

        self.thread = threading.Thread(
            target=self._sampling_loop,
            daemon=True
        )

        self.thread.start()

    def stop(
        self,
        elapsed_seconds,
        number_of_images
    ):
        self.stop_event.set()

        if self.thread is not None:
            self.thread.join()

        try:
            self._collect_sample()
        except Exception:
            pass

        data = pd.DataFrame(
            self.samples
        )

        nvmlShutdown()

        if data.empty:
            raise RuntimeError(
                "No resource samples were collected."
            )

        baseline_ram = data[
            "ram_mb"
        ].iloc[0]

        baseline_gpu_memory = data[
            "gpu_memory_mb"
        ].iloc[0]

        average_ram = data[
            "ram_mb"
        ].mean()

        peak_ram = data[
            "ram_mb"
        ].max()

        average_gpu_memory = data[
            "gpu_memory_mb"
        ].mean()

        peak_gpu_memory = data[
            "gpu_memory_mb"
        ].max()

        average_incremental_ram = max(
            0.0,
            average_ram - baseline_ram
        )

        peak_incremental_ram = max(
            0.0,
            peak_ram - baseline_ram
        )

        average_incremental_gpu_memory = max(
            0.0,
            average_gpu_memory
            - baseline_gpu_memory
        )

        peak_incremental_gpu_memory = max(
            0.0,
            peak_gpu_memory
            - baseline_gpu_memory
        )

        # Integrate GPU power over time
        valid_power = data.dropna(
            subset=["gpu_power_w"]
        )

        if len(valid_power) >= 2:
            relative_times = (
                valid_power[
                    "timestamp"
                ].to_numpy()
                - valid_power[
                    "timestamp"
                ].iloc[0]
            )

            # Compatibility with different NumPy versions
            if hasattr(np, "trapezoid"):
                energy_joules = np.trapezoid(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )
            else:
                energy_joules = np.trapz(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )

            energy_wh = (
                energy_joules / 3600.0
            )
        else:
            energy_wh = np.nan

        return {
            "elapsed_time_s":
                elapsed_seconds,

            "latency_ms_per_image":
                elapsed_seconds
                / number_of_images
                * 1000.0,

            "throughput_images_per_s":
                number_of_images
                / elapsed_seconds,

            "average_cpu_percent":
                data[
                    "cpu_percent"
                ].mean(),

            "peak_cpu_percent":
                data[
                    "cpu_percent"
                ].max(),

            "baseline_ram_mb":
                baseline_ram,

            "average_ram_mb":
                average_ram,

            "peak_ram_mb":
                peak_ram,

            "average_incremental_ram_mb":
                average_incremental_ram,

            "peak_incremental_ram_mb":
                peak_incremental_ram,

            "baseline_gpu_memory_mb":
                baseline_gpu_memory,

            "average_gpu_memory_mb":
                average_gpu_memory,

            "peak_gpu_memory_mb":
                peak_gpu_memory,

            "average_incremental_gpu_memory_mb":
                average_incremental_gpu_memory,

            "peak_incremental_gpu_memory_mb":
                peak_incremental_gpu_memory,

            "average_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].mean(),

            "peak_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].max(),

            "average_gpu_power_w":
                data[
                    "gpu_power_w"
                ].mean(),

            "peak_gpu_power_w":
                data[
                    "gpu_power_w"
                ].max(),

            "gpu_energy_wh":
                energy_wh
        }


print("ResourceMonitor defined successfully.")

ResourceMonitor defined successfully.


C:\Users\aneek\AppData\Local\Temp\ipykernel_30976\2400127911.py:13: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  from pynvml import (


In [30]:
# ============================================================
# PYTORCH MAXVIT WARM-UP AND REPEATED INFERENCE PROFILING
# ============================================================

import gc
import time
import torch
import pandas as pd


NUMBER_OF_RUNS = 5
WARMUP_BATCHES = 3
COOLDOWN_SECONDS = 5


# ============================================================
# DEVICE AND MODEL
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = model.to(device)
model.eval()

number_of_test_images = len(
    test_loader.dataset
)

print("Device:", device)
print("Test images:", number_of_test_images)
print("Test batches:", len(test_loader))


# ============================================================
# GPU/MODEL WARM-UP
# ============================================================

print(
    f"\nPerforming warm-up using "
    f"{WARMUP_BATCHES} batches..."
)

with torch.inference_mode():

    for batch_index, batch in enumerate(
        test_loader
    ):

        if batch_index >= WARMUP_BATCHES:
            break

        # MaxViT dataset returns:
        # images tensor, labels tensor
        images, labels = batch

        images = images.to(
            device,
            dtype=torch.float32,
            non_blocking=True
        )

        # timm MaxViT returns logits directly
        logits = model(images)

        # Materialize part of the output
        _ = logits[-1, 0].item()


if device.type == "cuda":
    torch.cuda.synchronize()

print("Warm-up completed.")


# ============================================================
# REPEATED INFERENCE RUNS
# ============================================================

run_results = []

for run_number in range(
    1,
    NUMBER_OF_RUNS + 1
):

    print(
        f"\nStarting MaxViT resource run "
        f"{run_number}/{NUMBER_OF_RUNS}"
    )

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    time.sleep(COOLDOWN_SECONDS)

    monitor = ResourceMonitor(
        interval=SAMPLING_INTERVAL,
        gpu_index=GPU_INDEX
    )

    monitor.start()

    # Ensure no previous CUDA operations remain queued
    if device.type == "cuda":
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    processed_images = 0
    last_logits = None

    with torch.inference_mode():

        for images, labels in test_loader:

            images = images.to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

            # timm MaxViT forward pass
            logits = model(images)

            last_logits = logits

            processed_images += images.size(0)


    # Wait for all CUDA inference operations
    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed_time = (
        time.perf_counter()
        - start_time
    )


    if last_logits is None:
        monitor.stop(
            elapsed_seconds=0.0,
            number_of_images=0
        )

        raise RuntimeError(
            "No images were processed during inference."
        )


    last_output_value = float(
        last_logits[-1, 0]
        .detach()
        .cpu()
        .item()
    )


    run_summary = monitor.stop(
        elapsed_seconds=elapsed_time,
        number_of_images=processed_images
    )

    run_summary["run"] = run_number

    run_summary[
        "processed_images"
    ] = processed_images

    run_summary[
        "last_output_value"
    ] = last_output_value

    run_results.append(
        run_summary
    )


    print(
        f"Run {run_number}: "
        f"{elapsed_time:.2f} seconds | "
        f"{run_summary['latency_ms_per_image']:.4f} "
        f"ms/image | "
        f"{run_summary['throughput_images_per_s']:.2f} "
        f"images/s"
    )


    del last_logits
    del logits
    del images


# ============================================================
# DISPLAY INDIVIDUAL RUNS
# ============================================================

results_df = pd.DataFrame(
    run_results
)

print(
    "\nIndividual MaxViT profiling runs:"
)

display(results_df)

Device: cuda
Test images: 18000
Test batches: 1125

Performing warm-up using 3 batches...


C:\Users\aneek\AppData\Local\Temp\ipykernel_30976\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(


Warm-up completed.

Starting MaxViT resource run 1/5
Run 1: 190.69 seconds | 10.5939 ms/image | 94.39 images/s

Starting MaxViT resource run 2/5


C:\Users\aneek\AppData\Local\Temp\ipykernel_30976\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(


Run 2: 186.94 seconds | 10.3855 ms/image | 96.29 images/s

Starting MaxViT resource run 3/5


...

C:\Users\aneek\AppData\Local\Temp\ipykernel_30976\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(


Run 4: 185.60 seconds | 10.3113 ms/image | 96.98 images/s

Starting MaxViT resource run 5/5


C:\Users\aneek\AppData\Local\Temp\ipykernel_30976\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(


Run 5: 190.14 seconds | 10.5633 ms/image | 94.67 images/s

Individual MaxViT profiling runs:


,elapsed_time_s,latency_ms_per_image,throughput_images_per_s,average_cpu_percent,peak_cpu_percent,baseline_ram_mb,average_ram_mb,peak_ram_mb,average_incremental_ram_mb,peak_incremental_ram_mb,...,average_incremental_gpu_memory_mb,peak_incremental_gpu_memory_mb,average_gpu_utilization_percent,peak_gpu_utilization_percent,average_gpu_power_w,peak_gpu_power_w,gpu_energy_wh,run,processed_images,last_output_value
0,190.690980,10.593943,94.393557,3.033982,12.428125,4807.570312,4808.041738,4812.683594,0.471426,5.113281,...,833.092067,860.000000,99.537348,100,50.600258,69.489,2.682051,1,18000,0.49009
1,186.939800,10.385544,96.287682,2.999222,9.406250,4808.675781,4803.992478,4813.609375,0.000000,4.933594,...,1.994090,2.000000,99.736998,100,51.843262,57.768,2.694282,2,18000,0.49009
2,186.395989,10.355333,96.568601,3.002666,11.096875,4803.695312,4803.969698,4808.605469,0.274385,4.910156,...,1.994083,2.000000,99.767456,100,52.152625,62.522,2.702827,3,18000,0.49009
3,185.603895,10.311328,96.980723,3.000451,11.646875,4803.796875,4804.031978,4808.656250,0.235103,4.859375,...,1.994062,2.000000,99.702494,100,52.204170,65.853,2.693534,4,18000,0.49009
4,190.139937,10.563330,94.667119,2.926212,8.878125,4803.894531,3894.462731,4806.660156,0.000000,2.765625,...,2.001255,2.011719,99.766957,100,51.286691,58.825,2.710810,5,18000,0.49009


In [31]:
# ============================================================RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# ============================================================
import numpy as np
import pandas as pd
from scipy.stats import t
metrics_to_report = [
    "elapsed_time_s",
    "latency_ms_per_image",
    "throughput_images_per_s",

    "average_cpu_percent",
    "peak_cpu_percent",

    "average_ram_mb",
    "peak_ram_mb",
    "average_incremental_ram_mb",
    "peak_incremental_ram_mb",

    "average_gpu_memory_mb",
    "peak_gpu_memory_mb",
    "average_incremental_gpu_memory_mb",
    "peak_incremental_gpu_memory_mb",

    "average_gpu_utilization_percent",
    "peak_gpu_utilization_percent",

    "average_gpu_power_w",
    "peak_gpu_power_w",
    "gpu_energy_wh"
]


summary_rows = []

number_of_runs = len(results_df)

for metric in metrics_to_report:
    values = results_df[metric].dropna()

    mean_value = values.mean()
    standard_deviation = values.std(ddof=1)

    if len(values) > 1:
        critical_t = t.ppf(
            0.975,
            df=len(values) - 1
        )

        confidence_half_width = (
            critical_t
            * standard_deviation
            / np.sqrt(len(values))
        )
    else:
        confidence_half_width = np.nan

    summary_rows.append({
        "Metric": metric,
        "Mean": mean_value,
        "Standard Deviation": standard_deviation,
        "95% CI Lower": (
            mean_value - confidence_half_width
        ),
        "95% CI Upper": (
            mean_value + confidence_half_width
        )
    })


resource_summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 90)
print("MaxViT RESOURCE CONSUMPTION — FIVE INFERENCE RUNS")
print("=" * 90)

display(resource_summary)


print("\nMain values for the manuscript")
print("-" * 90)

for metric in [
    "latency_ms_per_image",
    "peak_ram_mb",
    "peak_gpu_memory_mb",
    "average_gpu_utilization_percent",
    "average_gpu_power_w"
]:
    row = resource_summary[
        resource_summary["Metric"] == metric
    ].iloc[0]

    print(
        f"{metric}: "
        f"{row['Mean']:.3f} ± "
        f"{row['Standard Deviation']:.3f} "
        f"(95% CI: "
        f"{row['95% CI Lower']:.3f}–"
        f"{row['95% CI Upper']:.3f})"
    )


MaxViT RESOURCE CONSUMPTION — FIVE INFERENCE RUNS


,Metric,Mean,Standard Deviation,95% CI Lower,95% CI Upper
0,elapsed_time_s,187.954120,2.304798,185.092336,190.815905
1,latency_ms_per_image,10.441896,0.128044,10.282908,10.600884
2,throughput_images_per_s,95.779537,1.170694,94.325928,97.233145
3,average_cpu_percent,2.992507,0.039770,2.943126,3.041888
4,peak_cpu_percent,10.691250,1.502759,8.825329,12.557171
5,average_ram_mb,4622.899725,407.212424,4117.278718,5128.520731
6,peak_ram_mb,4810.042969,2.963316,4806.363526,4813.722412
7,average_incremental_ram_mb,0.196183,0.200228,-0.052433,0.444798
8,peak_incremental_ram_mb,4.516406,0.983388,3.295369,5.737444
9,average_gpu_memory_mb,7233.197142,12.031748,7218.257754,7248.136531



Main values for the manuscript
------------------------------------------------------------------------------------------
latency_ms_per_image: 10.442 ± 0.128 (95% CI: 10.283–10.601)
peak_ram_mb: 4810.043 ± 2.963 (95% CI: 4806.364–4813.722)
peak_gpu_memory_mb: 7238.584 ± 0.005 (95% CI: 7238.578–7238.591)
average_gpu_utilization_percent: 99.702 ± 0.096 (95% CI: 99.583–99.821)
average_gpu_power_w: 51.617 ± 0.676 (95% CI: 50.779–52.456)


genralization

In [33]:
print("\nTest results of wild deepfake dataset on Celeb-DF(V2) (MaxViT):")
# Dataloaders
test_dataset = DeepfakeMaxViTDataset(images=test_celeb,labels=test_labels,transform=maxvit_transform)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results = evaluate_complete(model=model,loader=test_loader,criterion=criterion,device=device)
print("\nMAXVIT-BASE TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if metric in [
        "confusion_matrix",
        "classification_report",
        "predictions"
    ]:
        continue

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )
    else:
        print(
            f"{metric:30s}: {value}"
        )


Test results of wild deepfake dataset on Celeb-DF(V2) (MaxViT):


C:\Users\aneek\AppData\Local\Temp\ipykernel_30976\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(



MAXVIT-BASE TEST RESULTS
test_loss                     : 0.924833
accuracy                      : 0.737796
balanced_accuracy             : 0.663201
precision                     : 0.943248
recall_sensitivity            : 0.755473
specificity                   : 0.570928
f1_score                      : 0.838982
mcc                           : 0.214943
roc_auc                       : 0.742548
pr_auc                        : 0.962817
average_precision             : 0.962823
eer                           : 0.332516
eer_threshold                 : 0.796132
false_positive_rate           : 0.429072
false_negative_rate           : 0.244527
true_negatives                : 326
false_positives               : 245
false_negatives               : 1318
true_positives                : 4072
number_of_test_images         : 5961


In [35]:
#dfc on wilddeepfake
print("\nTest results of wild deepfake dataset on DFC (MaxViT):")
# Dataloaders
test_dataset = DeepfakeMaxViTDataset(images=test_hog,labels=test_labels,transform=maxvit_transform)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results = evaluate_complete(model=model,loader=test_loader,criterion=criterion,device=device)
print("\nMAXVIT-BASE TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if metric in [
        "confusion_matrix",
        "classification_report",
        "predictions"
    ]:
        continue

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )
    else:
        print(
            f"{metric:30s}: {value}"
        )



Test results of wild deepfake dataset on DFC (MaxViT):


C:\Users\aneek\AppData\Local\Temp\ipykernel_30976\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(



MAXVIT-BASE TEST RESULTS
test_loss                     : 1.961395
accuracy                      : 0.445333
balanced_accuracy             : 0.445333
precision                     : 0.410675
recall_sensitivity            : 0.251333
specificity                   : 0.639333
f1_score                      : 0.311828
mcc                           : -0.118627
roc_auc                       : 0.415285
pr_auc                        : 0.443308
average_precision             : 0.444013
eer                           : 0.564667
eer_threshold                 : 0.143654
false_positive_rate           : 0.360667
false_negative_rate           : 0.748667
true_negatives                : 959
false_positives               : 541
false_negatives               : 1123
true_positives                : 377
number_of_test_images         : 3000


In [37]:
print("\nTest results of wild deepfake dataset on FF++ (MaxViT):")
#ff++ on wilddeepfake
# Dataloaders
test_dataset = DeepfakeMaxViTDataset(images=test_ff,labels=test_ff_labels,transform=maxvit_transform)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results = evaluate_complete(model=model,loader=test_loader,criterion=criterion,device=device)
print("\nMAXVIT-BASE TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if metric in [
        "confusion_matrix",
        "classification_report",
        "predictions"
    ]:
        continue

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )
    else:
        print(
            f"{metric:30s}: {value}"
        )


Test results of wild deepfake dataset on FF++ (MaxViT):


C:\Users\aneek\AppData\Local\Temp\ipykernel_30976\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(



MAXVIT-BASE TEST RESULTS
test_loss                     : 1.798889
accuracy                      : 0.532929
balanced_accuracy             : 0.548541
precision                     : 0.456867
recall_sensitivity            : 0.642095
specificity                   : 0.454986
f1_score                      : 0.533871
mcc                           : 0.097149
roc_auc                       : 0.571369
pr_auc                        : 0.497451
average_precision             : 0.498201
eer                           : 0.446076
eer_threshold                 : 0.716145
false_positive_rate           : 0.545014
false_negative_rate           : 0.357905
true_negatives                : 657
false_positives               : 787
false_negatives               : 369
true_positives                : 662
number_of_test_images         : 2475


# Celeb

In [22]:
import os
import cv2
import numpy as np

SAVE_ROOT = r'D:\thesis\celeb_processed'

def load_split(split_name, class_name):
    """Reload saved frames, grouped by video."""
    base = os.path.join(SAVE_ROOT, split_name, class_name)
    nested, ids = [], []
    for vid_id in sorted(os.listdir(base)):
        vid_dir = os.path.join(base, vid_id)
        frames = [cv2.imread(os.path.join(vid_dir, f))
                  for f in sorted(os.listdir(vid_dir))]
        if frames:
            nested.append(frames)
            ids.append(vid_id)
    return nested, ids

# Reload ALL six splits
print("Loading frames...")
real_train_final,  real_train_ids  = load_split('train', 'real')
synth_train_final, synth_train_ids = load_split('train', 'fake')
real_val_final,    real_val_ids    = load_split('val',   'real')
synth_val_final,   synth_val_ids   = load_split('val',   'fake')
real_test_final,   real_test_ids   = load_split('test',  'real')
synth_test_final,  synth_test_ids  = load_split('test',  'fake')

print("✅ All frames reloaded")
print("Train -> real videos:", len(real_train_final), " fake videos:", len(synth_train_final))
print("Val   -> real videos:", len(real_val_final),   " fake videos:", len(synth_val_final))
print("Test  -> real videos:", len(real_test_final),  " fake videos:", len(synth_test_final))
print("Example frame shape:", np.shape(real_train_final[0][0]))  # expect (160,160,3)
import numpy as np


def combine_split(real_videos, fake_videos):
    """
    Flatten video-grouped frames into one image array and create labels.

    real_videos: list of videos, where each video is a list of frames
    fake_videos: list of videos, where each video is a list of frames

    Returns
    -------
    images : NumPy array with shape (N, 160, 160, 3)
    labels : NumPy array with shape (N,)
             0 = real, 1 = fake
    """

    # Flatten frames from all real videos
    real_frames = [
        frame
        for video_frames in real_videos
        for frame in video_frames
        if frame is not None
    ]

    # Flatten frames from all fake videos
    fake_frames = [
        frame
        for video_frames in fake_videos
        for frame in video_frames
        if frame is not None
    ]

    if len(real_frames) == 0:
        raise ValueError("No real frames were found.")

    if len(fake_frames) == 0:
        raise ValueError("No fake frames were found.")

    # Convert to NumPy arrays
    real_frames = np.stack(real_frames).astype(np.uint8)
    fake_frames = np.stack(fake_frames).astype(np.uint8)

    # Combine images
    images = np.concatenate(
        [real_frames, fake_frames],
        axis=0
    )

    # Create labels
    real_labels = np.zeros(
        len(real_frames),
        dtype=np.uint8
    )

    fake_labels = np.ones(
        len(fake_frames),
        dtype=np.uint8
    )

    labels = np.concatenate(
        [real_labels, fake_labels],
        axis=0
    )

    return images, labels
# Training set
train_celeb, train_labels = combine_split(
    real_train_final,
    synth_train_final
)

# Validation set
val_celeb, val_labels = combine_split(
    real_val_final,
    synth_val_final
)

# Testing set
test_celeb, test_labels = combine_split(
    real_test_final,
    synth_test_final
)
print("\nTRAIN")
print("Images:", train_celeb.shape)
print("Labels:", train_labels.shape)
print("Real:", np.sum(train_labels == 0))
print("Fake:", np.sum(train_labels == 1))

print("\nVALIDATION")
print("Images:", val_celeb.shape)
print("Labels:", val_labels.shape)
print("Real:", np.sum(val_labels == 0))
print("Fake:", np.sum(val_labels == 1))

print("\nTEST")
print("Images:", test_celeb.shape)
print("Labels:", test_labels.shape)
print("Real:", np.sum(test_labels == 0))
print("Fake:", np.sum(test_labels == 1))

print("\nData types")
print("Train images:", train_celeb.dtype)
print("Train labels:", train_labels.dtype)

Loading frames...
✅ All frames reloaded
Train -> real videos: 354  fake videos: 3383
Val   -> real videos: 59  fake videos: 563
Test  -> real videos: 177  fake videos: 1693
Example frame shape: (160, 160, 3)

TRAIN
Images: (11899, 160, 160, 3)
Labels: (11899,)
Real: 1142
Fake: 10757

VALIDATION
Images: (1969, 160, 160, 3)
Labels: (1969,)
Real: 182
Fake: 1787

TEST
Images: (5961, 160, 160, 3)
Labels: (5961,)
Real: 571
Fake: 5390

Data types
Train images: uint8
Train labels: uint8


In [9]:
train_dataset = DeepfakeMaxViTDataset(images=train_celeb,labels=train_labels,transform=maxvit_transform)
val_dataset = DeepfakeMaxViTDataset(images=val_celeb,labels=val_labels,transform=maxvit_transform)
test_dataset = DeepfakeMaxViTDataset(images=test_celeb,labels=test_labels,transform=maxvit_transform)

train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())


print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Testing samples:", len(test_dataset))

Training samples: 11899
Validation samples: 1969
Testing samples: 5961


In [10]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    epochs=10
)

C:\Users\aneek\AppData\Local\Temp\ipykernel_36624\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(


Epoch 01/10 | Train Loss: 0.1521 | Train Accuracy: 0.9465 | Val Loss: 0.6633 | Val Accuracy: 0.7410
Epoch 02/10 | Train Loss: 0.0411 | Train Accuracy: 0.9836 | Val Loss: 0.0470 | Val Accuracy: 0.9817
Epoch 03/10 | Train Loss: 0.0190 | Train Accuracy: 0.9933 | Val Loss: 0.1809 | Val Accuracy: 0.9563
Epoch 04/10 | Train Loss: 0.0174 | Train Accuracy: 0.9938 | Val Loss: 0.0573 | Val Accuracy: 0.9777
Epoch 05/10 | Train Loss: 0.0152 | Train Accuracy: 0.9947 | Val Loss: 0.0171 | Val Accuracy: 0.9939
Epoch 06/10 | Train Loss: 0.0149 | Train Accuracy: 0.9954 | Val Loss: 0.0854 | Val Accuracy: 0.9644
Epoch 07/10 | Train Loss: 0.0151 | Train Accuracy: 0.9947 | Val Loss: 0.0286 | Val Accuracy: 0.9893
Epoch 08/10 | Train Loss: 0.0085 | Train Accuracy: 0.9971 | Val Loss: 0.0167 | Val Accuracy: 0.9934
Epoch 09/10 | Train Loss: 0.0104 | Train Accuracy: 0.9969 | Val Loss: 0.0576 | Val Accuracy: 0.9812
Epoch 10/10 | Train Loss: 0.0118 | Train Accuracy: 0.9958 | Val Loss: 0.0218 | Val Accuracy: 0.9919


In [11]:
print("=== DATA LOADING ===")
start = monitor.get_stats()

=== DATA LOADING ===


In [12]:
print("=== DATA LOADING ===")
start = monitor1.get_stats()

=== DATA LOADING ===


In [13]:
test_results = evaluate_complete(
    model=model,
    loader=test_loader,
    criterion=criterion,
    device=device
)
print("\nMAXVIT-BASE TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if metric in [
        "confusion_matrix",
        "classification_report",
        "predictions"
    ]:
        continue

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )
    else:
        print(
            f"{metric:30s}: {value}"
        )

C:\Users\aneek\AppData\Local\Temp\ipykernel_36624\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(



MAXVIT-BASE TEST RESULTS
test_loss                     : 0.026647
accuracy                      : 0.992451
balanced_accuracy             : 0.965293
precision                     : 0.992808
recall_sensitivity            : 0.998887
specificity                   : 0.931699
f1_score                      : 0.995838
mcc                           : 0.955780
roc_auc                       : 0.999085
pr_auc                        : 0.999901
average_precision             : 0.999901
eer                           : 0.013870
eer_threshold                 : 0.992158
false_positive_rate           : 0.068301
false_negative_rate           : 0.001113
true_negatives                : 532
false_positives               : 39
false_negatives               : 6
true_positives                : 5384
number_of_test_images         : 5961


In [14]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 9.3%
Time Usage: 81.8 s
GPU Memory Used: 0.0 MB
Power Consumption: 93W


In [15]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor1.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 9.7%
Time Usage: 83.9 s
GPU Memory Used: 1845.1 MB
Power Consumption: 93W


save the model

In [16]:
# ============================================================
# SAVE TIMM MAXVIT MODEL
# ============================================================

import os
import torch


SAVE_DIR = os.path.join(
    r"D:\thesis\results",
    "maxvit_celeb_160"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)


CHECKPOINT_PATH = os.path.join(
    SAVE_DIR,
    "training_checkpoint.pt"
)

WEIGHTS_PATH = os.path.join(
    SAVE_DIR,
    "maxvit_weights.pth"
)


# Save model weights only
torch.save(
    model.state_dict(),
    WEIGHTS_PATH
)


# Save full training checkpoint
torch.save(
    {
        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "history":
            history,

        "model_name":
            MODEL_NAME,

        "input_size":
            INPUT_SIZE,

        "num_classes":
            NUM_CLASSES,

        "epochs":
            EPOCHS,

        "batch_size":
            BATCH_SIZE,

        "learning_rate":
            LEARNING_RATE,

        "random_seed":
            RANDOM_SEED,

        "label_mapping": {
            0: "real",
            1: "fake"
        },

        "normalization_mean": [
            0.485,
            0.456,
            0.406
        ],

        "normalization_std": [
            0.229,
            0.224,
            0.225
        ]
    },

    CHECKPOINT_PATH
)


print("MaxViT model saved successfully.")
print("Weights:", WEIGHTS_PATH)
print("Checkpoint:", CHECKPOINT_PATH)

MaxViT model saved successfully.
Weights: D:\thesis\results\maxvit_celeb_160\maxvit_weights.pth
Checkpoint: D:\thesis\results\maxvit_celeb_160\training_checkpoint.pt


In [18]:
# ============================================================
# RESOURCE MONITOR DEFINITION
# RUN THIS BEFORE THE ViT PROFILING CELL
# ============================================================

import os
import time
import threading
import numpy as np
import pandas as pd
import psutil

from pynvml import (
    nvmlInit,
    nvmlShutdown,
    nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetMemoryInfo,
    nvmlDeviceGetUtilizationRates,
    nvmlDeviceGetPowerUsage,
    NVMLError
)


SAMPLING_INTERVAL = 0.1
GPU_INDEX = 0


class ResourceMonitor:
    """
    Continuously samples CPU, RAM, GPU memory,
    GPU utilization, and GPU power.
    """

    def __init__(
        self,
        interval=0.1,
        gpu_index=0
    ):
        self.interval = interval

        self.process = psutil.Process(
            os.getpid()
        )

        self.logical_cpu_count = (
            psutil.cpu_count(logical=True) or 1
        )

        self.stop_event = threading.Event()
        self.samples = []
        self.thread = None

        nvmlInit()

        self.gpu_handle = (
            nvmlDeviceGetHandleByIndex(
                gpu_index
            )
        )

    def _read_gpu_power(self):
        try:
            return (
                nvmlDeviceGetPowerUsage(
                    self.gpu_handle
                ) / 1000.0
            )
        except NVMLError:
            return np.nan

    def _collect_sample(self):
        timestamp = time.perf_counter()

        ram_mb = (
            self.process.memory_info().rss
            / (1024 ** 2)
        )

        process_cpu_raw = (
            self.process.cpu_percent(
                interval=None
            )
        )

        process_cpu_normalized = (
            process_cpu_raw
            / self.logical_cpu_count
        )

        gpu_memory = nvmlDeviceGetMemoryInfo(
            self.gpu_handle
        )

        gpu_memory_used_mb = (
            gpu_memory.used
            / (1024 ** 2)
        )

        gpu_utilization = (
            nvmlDeviceGetUtilizationRates(
                self.gpu_handle
            ).gpu
        )

        gpu_power_w = self._read_gpu_power()

        self.samples.append({
            "timestamp": timestamp,
            "ram_mb": ram_mb,
            "cpu_percent": process_cpu_normalized,
            "gpu_memory_mb": gpu_memory_used_mb,
            "gpu_utilization_percent": gpu_utilization,
            "gpu_power_w": gpu_power_w
        })

    def _sampling_loop(self):
        while not self.stop_event.is_set():
            try:
                self._collect_sample()
            except Exception as error:
                print(
                    "Monitoring warning:",
                    error
                )

            self.stop_event.wait(
                self.interval
            )

    def start(self):
        # Initialize the CPU utilization counter
        self.process.cpu_percent(
            interval=None
        )

        # First observation is the baseline
        self._collect_sample()

        self.thread = threading.Thread(
            target=self._sampling_loop,
            daemon=True
        )

        self.thread.start()

    def stop(
        self,
        elapsed_seconds,
        number_of_images
    ):
        self.stop_event.set()

        if self.thread is not None:
            self.thread.join()

        try:
            self._collect_sample()
        except Exception:
            pass

        data = pd.DataFrame(
            self.samples
        )

        nvmlShutdown()

        if data.empty:
            raise RuntimeError(
                "No resource samples were collected."
            )

        baseline_ram = data[
            "ram_mb"
        ].iloc[0]

        baseline_gpu_memory = data[
            "gpu_memory_mb"
        ].iloc[0]

        average_ram = data[
            "ram_mb"
        ].mean()

        peak_ram = data[
            "ram_mb"
        ].max()

        average_gpu_memory = data[
            "gpu_memory_mb"
        ].mean()

        peak_gpu_memory = data[
            "gpu_memory_mb"
        ].max()

        average_incremental_ram = max(
            0.0,
            average_ram - baseline_ram
        )

        peak_incremental_ram = max(
            0.0,
            peak_ram - baseline_ram
        )

        average_incremental_gpu_memory = max(
            0.0,
            average_gpu_memory
            - baseline_gpu_memory
        )

        peak_incremental_gpu_memory = max(
            0.0,
            peak_gpu_memory
            - baseline_gpu_memory
        )

        # Integrate GPU power over time
        valid_power = data.dropna(
            subset=["gpu_power_w"]
        )

        if len(valid_power) >= 2:
            relative_times = (
                valid_power[
                    "timestamp"
                ].to_numpy()
                - valid_power[
                    "timestamp"
                ].iloc[0]
            )

            # Compatibility with different NumPy versions
            if hasattr(np, "trapezoid"):
                energy_joules = np.trapezoid(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )
            else:
                energy_joules = np.trapz(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )

            energy_wh = (
                energy_joules / 3600.0
            )
        else:
            energy_wh = np.nan

        return {
            "elapsed_time_s":
                elapsed_seconds,

            "latency_ms_per_image":
                elapsed_seconds
                / number_of_images
                * 1000.0,

            "throughput_images_per_s":
                number_of_images
                / elapsed_seconds,

            "average_cpu_percent":
                data[
                    "cpu_percent"
                ].mean(),

            "peak_cpu_percent":
                data[
                    "cpu_percent"
                ].max(),

            "baseline_ram_mb":
                baseline_ram,

            "average_ram_mb":
                average_ram,

            "peak_ram_mb":
                peak_ram,

            "average_incremental_ram_mb":
                average_incremental_ram,

            "peak_incremental_ram_mb":
                peak_incremental_ram,

            "baseline_gpu_memory_mb":
                baseline_gpu_memory,

            "average_gpu_memory_mb":
                average_gpu_memory,

            "peak_gpu_memory_mb":
                peak_gpu_memory,

            "average_incremental_gpu_memory_mb":
                average_incremental_gpu_memory,

            "peak_incremental_gpu_memory_mb":
                peak_incremental_gpu_memory,

            "average_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].mean(),

            "peak_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].max(),

            "average_gpu_power_w":
                data[
                    "gpu_power_w"
                ].mean(),

            "peak_gpu_power_w":
                data[
                    "gpu_power_w"
                ].max(),

            "gpu_energy_wh":
                energy_wh
        }


print("ResourceMonitor defined successfully.")

ResourceMonitor defined successfully.


C:\Users\aneek\AppData\Local\Temp\ipykernel_36624\2400127911.py:13: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  from pynvml import (


In [19]:
# ============================================================
# PYTORCH MAXVIT WARM-UP AND REPEATED INFERENCE PROFILING
# ============================================================

import gc
import time
import torch
import pandas as pd


NUMBER_OF_RUNS = 5
WARMUP_BATCHES = 3
COOLDOWN_SECONDS = 5


# ============================================================
# DEVICE AND MODEL
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = model.to(device)
model.eval()

number_of_test_images = len(
    test_loader.dataset
)

print("Device:", device)
print("Test images:", number_of_test_images)
print("Test batches:", len(test_loader))


# ============================================================
# GPU/MODEL WARM-UP
# ============================================================

print(
    f"\nPerforming warm-up using "
    f"{WARMUP_BATCHES} batches..."
)

with torch.inference_mode():

    for batch_index, batch in enumerate(
        test_loader
    ):

        if batch_index >= WARMUP_BATCHES:
            break

        # MaxViT dataset returns:
        # images tensor, labels tensor
        images, labels = batch

        images = images.to(
            device,
            dtype=torch.float32,
            non_blocking=True
        )

        # timm MaxViT returns logits directly
        logits = model(images)

        # Materialize part of the output
        _ = logits[-1, 0].item()


if device.type == "cuda":
    torch.cuda.synchronize()

print("Warm-up completed.")


# ============================================================
# REPEATED INFERENCE RUNS
# ============================================================

run_results = []

for run_number in range(
    1,
    NUMBER_OF_RUNS + 1
):

    print(
        f"\nStarting MaxViT resource run "
        f"{run_number}/{NUMBER_OF_RUNS}"
    )

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    time.sleep(COOLDOWN_SECONDS)

    monitor = ResourceMonitor(
        interval=SAMPLING_INTERVAL,
        gpu_index=GPU_INDEX
    )

    monitor.start()

    # Ensure no previous CUDA operations remain queued
    if device.type == "cuda":
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    processed_images = 0
    last_logits = None

    with torch.inference_mode():

        for images, labels in test_loader:

            images = images.to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

            # timm MaxViT forward pass
            logits = model(images)

            last_logits = logits

            processed_images += images.size(0)


    # Wait for all CUDA inference operations
    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed_time = (
        time.perf_counter()
        - start_time
    )


    if last_logits is None:
        monitor.stop(
            elapsed_seconds=0.0,
            number_of_images=0
        )

        raise RuntimeError(
            "No images were processed during inference."
        )


    last_output_value = float(
        last_logits[-1, 0]
        .detach()
        .cpu()
        .item()
    )


    run_summary = monitor.stop(
        elapsed_seconds=elapsed_time,
        number_of_images=processed_images
    )

    run_summary["run"] = run_number

    run_summary[
        "processed_images"
    ] = processed_images

    run_summary[
        "last_output_value"
    ] = last_output_value

    run_results.append(
        run_summary
    )


    print(
        f"Run {run_number}: "
        f"{elapsed_time:.2f} seconds | "
        f"{run_summary['latency_ms_per_image']:.4f} "
        f"ms/image | "
        f"{run_summary['throughput_images_per_s']:.2f} "
        f"images/s"
    )


    del last_logits
    del logits
    del images


# ============================================================
# DISPLAY INDIVIDUAL RUNS
# ============================================================

results_df = pd.DataFrame(
    run_results
)

print(
    "\nIndividual MaxViT profiling runs:"
)

display(results_df)

Device: cuda
Test images: 5961
Test batches: 373

Performing warm-up using 3 batches...


C:\Users\aneek\AppData\Local\Temp\ipykernel_36624\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(


Warm-up completed.

Starting MaxViT resource run 1/5
Run 1: 65.03 seconds | 10.9097 ms/image | 91.66 images/s

Starting MaxViT resource run 2/5


C:\Users\aneek\AppData\Local\Temp\ipykernel_36624\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(


Run 2: 63.72 seconds | 10.6889 ms/image | 93.55 images/s

Starting MaxViT resource run 3/5


...

C:\Users\aneek\AppData\Local\Temp\ipykernel_36624\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(


Run 4: 66.43 seconds | 11.1448 ms/image | 89.73 images/s

Starting MaxViT resource run 5/5


C:\Users\aneek\AppData\Local\Temp\ipykernel_36624\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(


Run 5: 66.23 seconds | 11.1102 ms/image | 90.01 images/s

Individual MaxViT profiling runs:


,elapsed_time_s,latency_ms_per_image,throughput_images_per_s,average_cpu_percent,peak_cpu_percent,baseline_ram_mb,average_ram_mb,peak_ram_mb,average_incremental_ram_mb,peak_incremental_ram_mb,...,average_incremental_gpu_memory_mb,peak_incremental_gpu_memory_mb,average_gpu_utilization_percent,peak_gpu_utilization_percent,average_gpu_power_w,peak_gpu_power_w,gpu_energy_wh,run,processed_images,last_output_value
0,65.032600,10.909680,91.661720,2.825992,8.959375,2377.394531,2377.779294,2382.484375,0.384762,5.089844,...,683.891341,754.0,98.976231,100,48.548499,64.368,0.878956,1,5961,-4.931314
1,63.716711,10.688930,93.554735,2.828681,10.653125,2378.335938,2378.674371,2383.320312,0.338434,4.984375,...,1.993092,6.0,99.436960,100,50.355282,59.898,0.893058,2,5961,-4.931314
2,65.334619,10.960345,91.238001,2.898572,12.093750,2378.531250,2378.809002,2383.425781,0.277752,4.894531,...,1.989882,6.0,99.377740,100,48.325278,60.581,0.878933,3,5961,-4.931314
3,66.433930,11.144763,89.728246,2.934655,8.959375,2378.644531,2378.891715,2383.523438,0.247184,4.878906,...,2.000000,6.0,99.465116,100,47.113390,54.816,0.871358,4,5961,-4.931314
4,66.227639,11.110156,90.007738,2.808366,10.156250,2378.742188,2378.938903,2383.570312,0.196716,4.828125,...,1.986755,6.0,99.418874,100,47.879502,54.817,0.882748,5,5961,-4.931314


In [20]:
# ============================================================RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# ============================================================
import numpy as np
import pandas as pd
from scipy.stats import t
metrics_to_report = [
    "elapsed_time_s",
    "latency_ms_per_image",
    "throughput_images_per_s",

    "average_cpu_percent",
    "peak_cpu_percent",

    "average_ram_mb",
    "peak_ram_mb",
    "average_incremental_ram_mb",
    "peak_incremental_ram_mb",

    "average_gpu_memory_mb",
    "peak_gpu_memory_mb",
    "average_incremental_gpu_memory_mb",
    "peak_incremental_gpu_memory_mb",

    "average_gpu_utilization_percent",
    "peak_gpu_utilization_percent",

    "average_gpu_power_w",
    "peak_gpu_power_w",
    "gpu_energy_wh"
]


summary_rows = []

number_of_runs = len(results_df)

for metric in metrics_to_report:
    values = results_df[metric].dropna()

    mean_value = values.mean()
    standard_deviation = values.std(ddof=1)

    if len(values) > 1:
        critical_t = t.ppf(
            0.975,
            df=len(values) - 1
        )

        confidence_half_width = (
            critical_t
            * standard_deviation
            / np.sqrt(len(values))
        )
    else:
        confidence_half_width = np.nan

    summary_rows.append({
        "Metric": metric,
        "Mean": mean_value,
        "Standard Deviation": standard_deviation,
        "95% CI Lower": (
            mean_value - confidence_half_width
        ),
        "95% CI Upper": (
            mean_value + confidence_half_width
        )
    })


resource_summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 90)
print("MaxViT RESOURCE CONSUMPTION — FIVE INFERENCE RUNS")
print("=" * 90)

display(resource_summary)


print("\nMain values for the manuscript")
print("-" * 90)

for metric in [
    "latency_ms_per_image",
    "peak_ram_mb",
    "peak_gpu_memory_mb",
    "average_gpu_utilization_percent",
    "average_gpu_power_w"
]:
    row = resource_summary[
        resource_summary["Metric"] == metric
    ].iloc[0]

    print(
        f"{metric}: "
        f"{row['Mean']:.3f} ± "
        f"{row['Standard Deviation']:.3f} "
        f"(95% CI: "
        f"{row['95% CI Lower']:.3f}–"
        f"{row['95% CI Upper']:.3f})"
    )


MaxViT RESOURCE CONSUMPTION — FIVE INFERENCE RUNS


,Metric,Mean,Standard Deviation,95% CI Lower,95% CI Upper
0,elapsed_time_s,65.349100,1.085560,64.001199,66.697001
1,latency_ms_per_image,10.962775,0.182110,10.736655,11.188895
2,throughput_images_per_s,91.238088,1.528046,89.340768,93.135408
3,average_cpu_percent,2.859253,0.054456,2.791637,2.926870
4,peak_cpu_percent,10.164375,1.310103,8.537667,11.791083
5,average_ram_mb,2378.618657,0.479840,2378.022857,2379.214457
6,peak_ram_mb,2383.264844,0.446736,2382.710148,2383.819539
7,average_incremental_ram_mb,0.288970,0.074171,0.196874,0.381066
8,peak_incremental_ram_mb,4.935156,0.103224,4.806987,5.063326
9,average_gpu_memory_mb,7377.881589,29.561307,7341.176378,7414.586799



Main values for the manuscript
------------------------------------------------------------------------------------------
latency_ms_per_image: 10.963 ± 0.182 (95% CI: 10.737–11.189)
peak_ram_mb: 2383.265 ± 0.447 (95% CI: 2382.710–2383.820)
peak_gpu_memory_mb: 7395.109 ± 0.000 (95% CI: 7395.109–7395.109)
average_gpu_utilization_percent: 99.335 ± 0.203 (95% CI: 99.083–99.587)
average_gpu_power_w: 48.444 ± 1.201 (95% CI: 46.953–49.935)


#genralization

In [22]:
#wild deepfake on celeb
print("\nTest results of Celeb-DF(V2) on wild deepfake dataset (MaxViT):")
test_dataset = DeepfakeMaxViTDataset(images=test_images,labels=test_labels,transform=maxvit_transform)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results = evaluate_complete(model=model,loader=test_loader,criterion=criterion,device=device)
print("\nMAXVIT-BASE TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if metric in [
        "confusion_matrix",
        "classification_report",
        "predictions"
    ]:
        continue

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )
    else:
        print(
            f"{metric:30s}: {value}"
        )


Test results of Celeb-DF(V2) on wild deepfake dataset (MaxViT):


C:\Users\aneek\AppData\Local\Temp\ipykernel_36624\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(



MAXVIT-BASE TEST RESULTS
test_loss                     : 3.072667
accuracy                      : 0.340333
balanced_accuracy             : 0.540074
precision                     : 0.874654
recall_sensitivity            : 0.140593
specificity                   : 0.939556
f1_score                      : 0.242246
mcc                           : 0.106585
roc_auc                       : 0.558957
pr_auc                        : 0.804272
average_precision             : 0.804287
eer                           : 0.452889
eer_threshold                 : 0.008090
false_positive_rate           : 0.060444
false_negative_rate           : 0.859407
true_negatives                : 4228
false_positives               : 272
false_negatives               : 11602
true_positives                : 1898
number_of_test_images         : 18000


In [24]:
#DFC on celeb
print("\nTest results of Celeb-DF(V2) on DFC dataset (maxvit):")
test_dataset = DeepfakeMaxViTDataset(images=test_hog,labels=test_labels,transform=maxvit_transform)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results = evaluate_complete(model=model,loader=test_loader,criterion=criterion,device=device)
print("\nMAXVIT-BASE TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if metric in [
        "confusion_matrix",
        "classification_report",
        "predictions"
    ]:
        continue

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )
    else:
        print(
            f"{metric:30s}: {value}"
        )


Test results of Celeb-DF(V2) on DFC dataset (maxvit):


C:\Users\aneek\AppData\Local\Temp\ipykernel_36624\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(



MAXVIT-BASE TEST RESULTS
test_loss                     : 2.676350
accuracy                      : 0.443667
balanced_accuracy             : 0.443667
precision                     : 0.446485
recall_sensitivity            : 0.470000
specificity                   : 0.417333
f1_score                      : 0.457941
mcc                           : -0.112823
roc_auc                       : 0.410633
pr_auc                        : 0.441731
average_precision             : 0.442620
eer                           : 0.563667
eer_threshold                 : 0.620265
false_positive_rate           : 0.582667
false_negative_rate           : 0.530000
true_negatives                : 626
false_positives               : 874
false_negatives               : 795
true_positives                : 705
number_of_test_images         : 3000


In [26]:
print("\nTest results of wild deepfake dataset on FF++ (MaxViT):")
#ff++ on wilddeepfake
test_dataset = DeepfakeMaxViTDataset(images=test_ff,labels=test_ff_labels,transform=maxvit_transform)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results = evaluate_complete(model=model,loader=test_loader,criterion=criterion,device=device)
print("\nMAXVIT-BASE TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if metric in [
        "confusion_matrix",
        "classification_report",
        "predictions"
    ]:
        continue

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )
    else:
        print(
            f"{metric:30s}: {value}"
        )


Test results of wild deepfake dataset on FF++ (MaxViT):


C:\Users\aneek\AppData\Local\Temp\ipykernel_36624\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(



MAXVIT-BASE TEST RESULTS
test_loss                     : 0.580630
accuracy                      : 0.806869
balanced_accuracy             : 0.815762
precision                     : 0.723164
recall_sensitivity            : 0.869059
specificity                   : 0.762465
f1_score                      : 0.789427
mcc                           : 0.622670
roc_auc                       : 0.920973
pr_auc                        : 0.912786
average_precision             : 0.912822
eer                           : 0.163677
eer_threshold                 : 0.774326
false_positive_rate           : 0.237535
false_negative_rate           : 0.130941
true_negatives                : 1101
false_positives               : 343
false_negatives               : 135
true_positives                : 896
number_of_test_images         : 2475


# DFC

In [8]:
import h5py
import numpy as np
# Open the HDF5 file in read mode
with h5py.File('D://thesis//dataset//deepfake dataset//resized_images.h5', 'r') as h5f:
    # Access each dataset
    celeb = np.array(h5f['celeb'])
    ffhq = np.array(h5f['ffhq'])
    gdwct = np.array(h5f['gdwct'])
    attgan = np.array(h5f['attgan'])
    stargan = np.array(h5f['stargan'])
    stylegan2 = np.array(h5f['stylegan2'])
    stylegan = np.array(h5f['stylegan'])

# Now, 'celeb', 'ffhq', etc., are NumPy arrays containing your datasets
print(f"celeb shape: {celeb.shape}, dtype: {celeb.dtype}")
print(f"ffhq shape: {ffhq.shape}, dtype: {ffhq.dtype}")
print(f"ffhq shape: {gdwct.shape}, dtype: {gdwct.dtype}")
print(f"ffhq shape: {attgan.shape}, dtype: {attgan.dtype}")
print(f"ffhq shape: {stargan.shape}, dtype: {stargan.dtype}")
print(f"ffhq shape: {stylegan2.shape}, dtype: {stylegan2.dtype}")
print(f"ffhq shape: {stylegan.shape}, dtype: {stylegan.dtype}")
# Repeat for other datasets as needed
import cv2
# Function to resize images from (224, 224) to (160, 160)
def resize_images(image_array, target_size=(160, 160)):
    resized_images = np.array([cv2.resize(img, target_size) for img in image_array])
    return resized_images

celeb = resize_images(celeb, target_size=(160, 160))
ffhq = resize_images(ffhq, target_size=(160, 160))
gdwct = resize_images(gdwct, target_size=(160, 160))
attgan = resize_images(attgan, target_size=(160, 160))
stargan = resize_images(stargan, target_size=(160, 160))
stylegan = resize_images(stylegan, target_size=(160, 160))
stylegan2 = resize_images(stylegan2, target_size=(160, 160))
import random
# Randomly select 2500 distinct images
random_indices = random.sample(range(len(celeb)), 2500)  # Get 2500 random indices
celeb = celeb[random_indices]  # Select the random subse

import random
# Randomly select 2500 distinct images
random_indices = random.sample(range(len(ffhq)), 2500)  # Get 2500 random indices
ffhq = ffhq[random_indices]  # Select the random subse
print(f"celeb shape: {celeb.shape}, dtype: {celeb.dtype}")
print(f"ffhq shape: {ffhq.shape}, dtype: {ffhq.dtype}")
print(f"gdwct shape: {gdwct.shape}, dtype: {gdwct.dtype}")
print(f"attagan shape: {attgan.shape}, dtype: {attgan.dtype}")
print(f"stargan shape: {stargan.shape}, dtype: {stargan.dtype}")
print(f"stylegan2 shape: {stylegan2.shape}, dtype: {stylegan2.dtype}")
print(f"stylegan shape: {stylegan.shape}, dtype: {stylegan.dtype}")
import random
import numpy as np

def split_data(data, train_ratio=0.7):
    """
    Splits data into training and testing sets based on the specified ratio.

    Parameters:
        data (list or np.array): The dataset to split.
        train_ratio (float): The ratio of the data to include in the training set.

    Returns:
        tuple: Two datasets - train and test.
    """
    # Shuffle the data
    random.shuffle(data)

    # Calculate the split index
    split_index = int(len(data) * train_ratio)

    # Split the data
    train_data = data[:split_index]
    test_data = data[split_index:]

    return train_data, test_data

# Split `celeb` into 70% train and 30% test
celeb_train_hog, celeb_test_hog = split_data(celeb, train_ratio=0.7)

# Split `ffhq` into 70% train and 30% test
ffhq_train_hog, ffhq_test_hog = split_data(ffhq, train_ratio=0.7)

# Split `attgan` into 70% train and 30% test
attgan_train_hog, attgan_test_hog = split_data(attgan, train_ratio=0.7)

# Split `stargan` into 70% train and 30% test
stargan_train_hog, stargan_test_hog = split_data(stargan, train_ratio=0.7)

# Split `gdwct` into 70% train and 30% test
gdwct_train_hog, gdwct_test_hog = split_data(gdwct, train_ratio=0.7)

# Split `stylegan2` into 70% train and 30% test_hog
stylegan2_train_hog, stylegan2_test_hog = split_data(stylegan2, train_ratio=0.7)

# Split `stylegan` into 70% train and 30% test_hog
stylegan_train_hog, stylegan_test_hog = split_data(stylegan, train_ratio=0.7)

# Convert to NumPy arrays if needed
celeb_train_hog, celeb_test_hog = np.array(celeb_train_hog), np.array(celeb_test_hog)
ffhq_train_hog, ffhq_test_hog = np.array(ffhq_train_hog), np.array(ffhq_test_hog)
attgan_train_hog, attgan_test_hog = np.array(attgan_train_hog), np.array(attgan_test_hog)
stargan_train_hog, stargan_test_hog = np.array(stargan_train_hog), np.array(stargan_test_hog)
gdwct_train_hog, gdwct_test_hog = np.array(gdwct_train_hog), np.array(gdwct_test_hog)
stylegan2_train_hog, stylegan2_test_hog = np.array(stylegan2_train_hog), np.array(stylegan2_test_hog)
stylegan_train_hog, stylegan_test_hog = np.array(stylegan_train_hog), np.array(stylegan_test_hog)

# Print results for verification
print(f"celeb_train: {len(celeb_train_hog)} images, celeb_test: {len(celeb_test_hog)} images")
print(f"ffhq_train: {len(ffhq_train_hog)} images, ffhq_test: {len(ffhq_test_hog)} images")
print(f"attgan_train: {len(attgan_train_hog)} images, attgan_test: {len(attgan_test_hog)} images")
print(f"stargan_train: {len(stargan_train_hog)} images, stargan_test: {len(stargan_test_hog)} images")
print(f"gdwct_train: {len(gdwct_train_hog)} images, gdwct_test: {len(gdwct_test_hog)} images")
print(f"stylegan2_train: {len(stylegan2_train_hog)} images, stylegan2_test: {len(stylegan2_test_hog)} images")
print(f"stylegan_train: {len(stylegan_train_hog)} images, stylegan_test: {len(stylegan_test_hog)} images")

########################################################################################################################################
#######################################divide into 60,10 train and val
#########################################################################################################################################
def extract_validation(train_data):
    """
    Extract every 10th sample from the training data and store it in a validation set.

    Parameters:
        train_data (list or np.array): The training dataset.

    Returns:
        tuple: Updated training dataset and validation dataset.
    """
    # Select every 10th sample for the validation set
    validation_data = train_data[::10]

    # Remove the selected samples from the training dataset
    updated_train_data = [train_data[i] for i in range(len(train_data)) if i % 10 != 0]

    return np.array(updated_train_data), np.array(validation_data)


# Perform the operation for each dataset
celeb_train_hog, celeb_val_hog = extract_validation(celeb_train_hog)
ffhq_train_hog, ffhq_val_hog = extract_validation(ffhq_train_hog)
attgan_train_hog, attgan_val_hog = extract_validation(attgan_train_hog)
stargan_train_hog, stargan_val_hog = extract_validation(stargan_train_hog)
gdwct_train_hog, gdwct_val_hog = extract_validation(gdwct_train_hog)
stylegan2_train_hog, stylegan2_val_hog = extract_validation(stylegan2_train_hog)
stylegan_train_hog, stylegan_val_hog = extract_validation(stylegan_train_hog)

# Print results for verification
print(f"celeb_train: {len(celeb_train_hog)} images, celeb_val: {len(celeb_val_hog)} images")
print(f"ffhq_train: {len(ffhq_train_hog)} images, ffhq_val: {len(ffhq_val_hog)} images")
print(f"attgan_train: {len(attgan_train_hog)} images, attgan_val: {len(attgan_val_hog)} images")
print(f"stargan_train: {len(stargan_train_hog)} images, stargan_val: {len(stargan_val_hog)} images")
print(f"gdwct_train: {len(gdwct_train_hog)} images, gdwct_val: {len(gdwct_val_hog)} images")
print(f"stylegan2_train: {len(stylegan2_train_hog)} images, stylegan2_val: {len(stylegan2_val_hog)} images")
print(f"stylegan_train: {len(stylegan_train_hog)} images, stylegan_val: {len(stylegan_val_hog)} images")
############################################################################################################################################################
#################################################concatenate the labels 0,1 real and fake
#############################################################################################################################################################


celeb_train_labels = np.zeros(len(celeb_train_hog), dtype=int)
ffhq_train_labels = np.zeros(len(ffhq_train_hog), dtype=int)
atta_train_labels = np.ones(len(attgan_train_hog), dtype=int)
star_train_labels = np.ones(len(stargan_train_hog), dtype=int)
gdwct_train_labels = np.ones(len(gdwct_train_hog), dtype=int)
stylegan2_train_labels = np.ones(len(stylegan2_train_hog), dtype=int)
stylegan_train_labels = np.ones(len(stylegan_train_hog), dtype=int)

# Concatenate all training datasets into a single `train` variable
train_hog = np.concatenate([celeb_train_hog, ffhq_train_hog, attgan_train_hog, stargan_train_hog, gdwct_train_hog, stylegan2_train_hog, stylegan_train_hog], axis=0)
train_labels=np.concatenate([celeb_train_labels, ffhq_train_labels, atta_train_labels, star_train_labels, gdwct_train_labels, stylegan2_train_labels,
                              stylegan_train_labels], axis=0)




celeb_test_labels = np.zeros(len(celeb_test_hog), dtype=int)
ffhq_test_labels = np.zeros(len(ffhq_test_hog), dtype=int)
atta_test_labels = np.ones(len(attgan_test_hog), dtype=int)
star_test_labels = np.ones(len(stargan_test_hog), dtype=int)
gdwct_test_labels = np.ones(len(gdwct_test_hog), dtype=int)
stylegan2_test_labels = np.ones(len(stylegan2_test_hog), dtype=int)
stylegan_test_labels = np.ones(len(stylegan_test_hog), dtype=int)

# Concatenate all testing datasets into a single `test` variable
test_hog = np.concatenate([celeb_test_hog, ffhq_test_hog, attgan_test_hog, stargan_test_hog, gdwct_test_hog, stylegan2_test_hog, stylegan_test_hog], axis=0)
test_labels = np.concatenate([celeb_test_labels, ffhq_test_labels, atta_test_labels, star_test_labels, gdwct_test_labels, stylegan2_test_labels,
                        stylegan_test_labels], axis=0)




celeb_val_labels = np.zeros(len(celeb_val_hog), dtype=int)
ffhq_val_labels = np.zeros(len(ffhq_val_hog), dtype=int)
atta_val_labels = np.ones(len(attgan_val_hog), dtype=int)
star_val_labels = np.ones(len(stargan_val_hog), dtype=int)
gdwct_val_labels = np.ones(len(gdwct_val_hog), dtype=int)
stylegan2_val_labels = np.ones(len(stylegan2_val_hog), dtype=int)
stylegan_val_labels = np.ones(len(stylegan_val_hog), dtype=int)

# Concatenate all validation datasets into a single `val` variable
val_hog = np.concatenate([celeb_val_hog, ffhq_val_hog, attgan_val_hog, stargan_val_hog, gdwct_val_hog, stylegan2_val_hog, stylegan_val_hog], axis=0)
val_labels = np.concatenate([celeb_val_labels, ffhq_val_labels, atta_val_labels, star_val_labels, gdwct_val_labels, stylegan2_val_labels,
                       stylegan_val_labels], axis=0)

# Print the results for verification
print(f"Total train: {len(train_hog)} images")
print(f"Total test: {len(test_hog)} images")
print(f"Total val: {len(val_hog)} images")


# Print results for verification
print(f"Train Labels: {len(train_labels)} ")
print(f"Test Labels: {len(test_labels)} ")
print(f"Val Labels: {len(val_labels)} ")



celeb shape: (5000, 224, 224, 3), dtype: uint8
ffhq shape: (5000, 224, 224, 3), dtype: uint8
ffhq shape: (1000, 224, 224, 3), dtype: uint8
ffhq shape: (1000, 224, 224, 3), dtype: uint8
ffhq shape: (1000, 224, 224, 3), dtype: uint8
ffhq shape: (1000, 224, 224, 3), dtype: uint8
ffhq shape: (1000, 224, 224, 3), dtype: uint8
celeb shape: (2500, 160, 160, 3), dtype: uint8
ffhq shape: (2500, 160, 160, 3), dtype: uint8
gdwct shape: (1000, 160, 160, 3), dtype: uint8
attagan shape: (1000, 160, 160, 3), dtype: uint8
stargan shape: (1000, 160, 160, 3), dtype: uint8
stylegan2 shape: (1000, 160, 160, 3), dtype: uint8
stylegan shape: (1000, 160, 160, 3), dtype: uint8
celeb_train: 1750 images, celeb_test: 750 images
ffhq_train: 1750 images, ffhq_test: 750 images
attgan_train: 700 images, attgan_test: 300 images
stargan_train: 700 images, stargan_test: 300 images
gdwct_train: 700 images, gdwct_test: 300 images
stylegan2_train: 700 images, stylegan2_test: 300 images
stylegan_train: 700 images, stylegan

In [9]:
train_dataset = DeepfakeMaxViTDataset(images=train_hog,labels=train_labels,transform=maxvit_transform)
val_dataset = DeepfakeMaxViTDataset(images=val_hog,labels=val_labels,transform=maxvit_transform)
test_dataset = DeepfakeMaxViTDataset(images=test_hog,labels=test_labels,transform=maxvit_transform)

train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())


print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Testing samples:", len(test_dataset))

Training samples: 6300
Validation samples: 700
Testing samples: 3000


In [10]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    epochs=10
)

C:\Users\aneek\AppData\Local\Temp\ipykernel_41544\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(


Epoch 01/10 | Train Loss: 0.1203 | Train Accuracy: 0.9562 | Val Loss: 0.0482 | Val Accuracy: 0.9843
Epoch 02/10 | Train Loss: 0.0253 | Train Accuracy: 0.9911 | Val Loss: 0.0373 | Val Accuracy: 0.9857
Epoch 03/10 | Train Loss: 0.0155 | Train Accuracy: 0.9943 | Val Loss: 0.0143 | Val Accuracy: 0.9957
Epoch 04/10 | Train Loss: 0.0181 | Train Accuracy: 0.9941 | Val Loss: 0.0203 | Val Accuracy: 0.9957
Epoch 05/10 | Train Loss: 0.0118 | Train Accuracy: 0.9956 | Val Loss: 0.0138 | Val Accuracy: 0.9957
Epoch 06/10 | Train Loss: 0.0041 | Train Accuracy: 0.9987 | Val Loss: 0.0167 | Val Accuracy: 0.9957
Epoch 07/10 | Train Loss: 0.0090 | Train Accuracy: 0.9963 | Val Loss: 0.0233 | Val Accuracy: 0.9943
Epoch 08/10 | Train Loss: 0.0085 | Train Accuracy: 0.9978 | Val Loss: 0.0045 | Val Accuracy: 0.9971
Epoch 09/10 | Train Loss: 0.0073 | Train Accuracy: 0.9979 | Val Loss: 0.0083 | Val Accuracy: 0.9986
Epoch 10/10 | Train Loss: 0.0003 | Train Accuracy: 1.0000 | Val Loss: 0.0112 | Val Accuracy: 0.9986


In [11]:
print("=== DATA LOADING ===")
start = monitor.get_stats()

=== DATA LOADING ===


In [12]:
print("=== DATA LOADING ===")
start = monitor1.get_stats()

=== DATA LOADING ===


In [13]:
test_results = evaluate_complete(
    model=model,
    loader=test_loader,
    criterion=criterion,
    device=device
)
print("\nMAXVIT-BASE TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if metric in [
        "confusion_matrix",
        "classification_report",
        "predictions"
    ]:
        continue

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )
    else:
        print(
            f"{metric:30s}: {value}"
        )

C:\Users\aneek\AppData\Local\Temp\ipykernel_41544\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(



MAXVIT-BASE TEST RESULTS
test_loss                     : 0.011618
accuracy                      : 0.996667
balanced_accuracy             : 0.996667
precision                     : 0.996667
recall_sensitivity            : 0.996667
specificity                   : 0.996667
f1_score                      : 0.996667
mcc                           : 0.993333
roc_auc                       : 0.999899
pr_auc                        : 0.999896
average_precision             : 0.999896
eer                           : 0.003333
eer_threshold                 : 0.766652
false_positive_rate           : 0.003333
false_negative_rate           : 0.003333
true_negatives                : 1495
false_positives               : 5
false_negatives               : 5
true_positives                : 1495
number_of_test_images         : 3000


In [14]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 15.2%
Time Usage: 39.9 s
GPU Memory Used: 0.0 MB
Power Consumption: 93W


In [15]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor1.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 11.9%
Time Usage: 42.1 s
GPU Memory Used: 1845.1 MB
Power Consumption: 93W


save the model

In [16]:
# ============================================================
# SAVE TIMM MAXVIT MODEL
# ============================================================

import os
import torch


SAVE_DIR = os.path.join(
    r"D:\thesis\results",
    "maxvit_dfc_160"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)


CHECKPOINT_PATH = os.path.join(
    SAVE_DIR,
    "training_checkpoint.pt"
)

WEIGHTS_PATH = os.path.join(
    SAVE_DIR,
    "maxvit_weights.pth"
)


# Save model weights only
torch.save(
    model.state_dict(),
    WEIGHTS_PATH
)


# Save full training checkpoint
torch.save(
    {
        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "history":
            history,

        "model_name":
            MODEL_NAME,

        "input_size":
            INPUT_SIZE,

        "num_classes":
            NUM_CLASSES,

        "epochs":
            EPOCHS,

        "batch_size":
            BATCH_SIZE,

        "learning_rate":
            LEARNING_RATE,

        "random_seed":
            RANDOM_SEED,

        "label_mapping": {
            0: "real",
            1: "fake"
        },

        "normalization_mean": [
            0.485,
            0.456,
            0.406
        ],

        "normalization_std": [
            0.229,
            0.224,
            0.225
        ]
    },

    CHECKPOINT_PATH
)


print("MaxViT model saved successfully.")
print("Weights:", WEIGHTS_PATH)
print("Checkpoint:", CHECKPOINT_PATH)

MaxViT model saved successfully.
Weights: D:\thesis\results\maxvit_dfc_160\maxvit_weights.pth
Checkpoint: D:\thesis\results\maxvit_dfc_160\training_checkpoint.pt


#load the model

In [17]:
# ============================================================
# RESOURCE MONITOR DEFINITION
# RUN THIS BEFORE THE ViT PROFILING CELL
# ============================================================

import os
import time
import threading
import numpy as np
import pandas as pd
import psutil

from pynvml import (
    nvmlInit,
    nvmlShutdown,
    nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetMemoryInfo,
    nvmlDeviceGetUtilizationRates,
    nvmlDeviceGetPowerUsage,
    NVMLError
)


SAMPLING_INTERVAL = 0.1
GPU_INDEX = 0


class ResourceMonitor:
    """
    Continuously samples CPU, RAM, GPU memory,
    GPU utilization, and GPU power.
    """

    def __init__(
        self,
        interval=0.1,
        gpu_index=0
    ):
        self.interval = interval

        self.process = psutil.Process(
            os.getpid()
        )

        self.logical_cpu_count = (
            psutil.cpu_count(logical=True) or 1
        )

        self.stop_event = threading.Event()
        self.samples = []
        self.thread = None

        nvmlInit()

        self.gpu_handle = (
            nvmlDeviceGetHandleByIndex(
                gpu_index
            )
        )

    def _read_gpu_power(self):
        try:
            return (
                nvmlDeviceGetPowerUsage(
                    self.gpu_handle
                ) / 1000.0
            )
        except NVMLError:
            return np.nan

    def _collect_sample(self):
        timestamp = time.perf_counter()

        ram_mb = (
            self.process.memory_info().rss
            / (1024 ** 2)
        )

        process_cpu_raw = (
            self.process.cpu_percent(
                interval=None
            )
        )

        process_cpu_normalized = (
            process_cpu_raw
            / self.logical_cpu_count
        )

        gpu_memory = nvmlDeviceGetMemoryInfo(
            self.gpu_handle
        )

        gpu_memory_used_mb = (
            gpu_memory.used
            / (1024 ** 2)
        )

        gpu_utilization = (
            nvmlDeviceGetUtilizationRates(
                self.gpu_handle
            ).gpu
        )

        gpu_power_w = self._read_gpu_power()

        self.samples.append({
            "timestamp": timestamp,
            "ram_mb": ram_mb,
            "cpu_percent": process_cpu_normalized,
            "gpu_memory_mb": gpu_memory_used_mb,
            "gpu_utilization_percent": gpu_utilization,
            "gpu_power_w": gpu_power_w
        })

    def _sampling_loop(self):
        while not self.stop_event.is_set():
            try:
                self._collect_sample()
            except Exception as error:
                print(
                    "Monitoring warning:",
                    error
                )

            self.stop_event.wait(
                self.interval
            )

    def start(self):
        # Initialize the CPU utilization counter
        self.process.cpu_percent(
            interval=None
        )

        # First observation is the baseline
        self._collect_sample()

        self.thread = threading.Thread(
            target=self._sampling_loop,
            daemon=True
        )

        self.thread.start()

    def stop(
        self,
        elapsed_seconds,
        number_of_images
    ):
        self.stop_event.set()

        if self.thread is not None:
            self.thread.join()

        try:
            self._collect_sample()
        except Exception:
            pass

        data = pd.DataFrame(
            self.samples
        )

        nvmlShutdown()

        if data.empty:
            raise RuntimeError(
                "No resource samples were collected."
            )

        baseline_ram = data[
            "ram_mb"
        ].iloc[0]

        baseline_gpu_memory = data[
            "gpu_memory_mb"
        ].iloc[0]

        average_ram = data[
            "ram_mb"
        ].mean()

        peak_ram = data[
            "ram_mb"
        ].max()

        average_gpu_memory = data[
            "gpu_memory_mb"
        ].mean()

        peak_gpu_memory = data[
            "gpu_memory_mb"
        ].max()

        average_incremental_ram = max(
            0.0,
            average_ram - baseline_ram
        )

        peak_incremental_ram = max(
            0.0,
            peak_ram - baseline_ram
        )

        average_incremental_gpu_memory = max(
            0.0,
            average_gpu_memory
            - baseline_gpu_memory
        )

        peak_incremental_gpu_memory = max(
            0.0,
            peak_gpu_memory
            - baseline_gpu_memory
        )

        # Integrate GPU power over time
        valid_power = data.dropna(
            subset=["gpu_power_w"]
        )

        if len(valid_power) >= 2:
            relative_times = (
                valid_power[
                    "timestamp"
                ].to_numpy()
                - valid_power[
                    "timestamp"
                ].iloc[0]
            )

            # Compatibility with different NumPy versions
            if hasattr(np, "trapezoid"):
                energy_joules = np.trapezoid(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )
            else:
                energy_joules = np.trapz(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )

            energy_wh = (
                energy_joules / 3600.0
            )
        else:
            energy_wh = np.nan

        return {
            "elapsed_time_s":
                elapsed_seconds,

            "latency_ms_per_image":
                elapsed_seconds
                / number_of_images
                * 1000.0,

            "throughput_images_per_s":
                number_of_images
                / elapsed_seconds,

            "average_cpu_percent":
                data[
                    "cpu_percent"
                ].mean(),

            "peak_cpu_percent":
                data[
                    "cpu_percent"
                ].max(),

            "baseline_ram_mb":
                baseline_ram,

            "average_ram_mb":
                average_ram,

            "peak_ram_mb":
                peak_ram,

            "average_incremental_ram_mb":
                average_incremental_ram,

            "peak_incremental_ram_mb":
                peak_incremental_ram,

            "baseline_gpu_memory_mb":
                baseline_gpu_memory,

            "average_gpu_memory_mb":
                average_gpu_memory,

            "peak_gpu_memory_mb":
                peak_gpu_memory,

            "average_incremental_gpu_memory_mb":
                average_incremental_gpu_memory,

            "peak_incremental_gpu_memory_mb":
                peak_incremental_gpu_memory,

            "average_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].mean(),

            "peak_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].max(),

            "average_gpu_power_w":
                data[
                    "gpu_power_w"
                ].mean(),

            "peak_gpu_power_w":
                data[
                    "gpu_power_w"
                ].max(),

            "gpu_energy_wh":
                energy_wh
        }


print("ResourceMonitor defined successfully.")

ResourceMonitor defined successfully.


C:\Users\aneek\AppData\Local\Temp\ipykernel_41544\2400127911.py:13: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  from pynvml import (


In [18]:
# ============================================================
# PYTORCH MAXVIT WARM-UP AND REPEATED INFERENCE PROFILING
# ============================================================

import gc
import time
import torch
import pandas as pd


NUMBER_OF_RUNS = 5
WARMUP_BATCHES = 3
COOLDOWN_SECONDS = 5


# ============================================================
# DEVICE AND MODEL
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = model.to(device)
model.eval()

number_of_test_images = len(
    test_loader.dataset
)

print("Device:", device)
print("Test images:", number_of_test_images)
print("Test batches:", len(test_loader))


# ============================================================
# GPU/MODEL WARM-UP
# ============================================================

print(
    f"\nPerforming warm-up using "
    f"{WARMUP_BATCHES} batches..."
)

with torch.inference_mode():

    for batch_index, batch in enumerate(
        test_loader
    ):

        if batch_index >= WARMUP_BATCHES:
            break

        # MaxViT dataset returns:
        # images tensor, labels tensor
        images, labels = batch

        images = images.to(
            device,
            dtype=torch.float32,
            non_blocking=True
        )

        # timm MaxViT returns logits directly
        logits = model(images)

        # Materialize part of the output
        _ = logits[-1, 0].item()


if device.type == "cuda":
    torch.cuda.synchronize()

print("Warm-up completed.")


# ============================================================
# REPEATED INFERENCE RUNS
# ============================================================

run_results = []

for run_number in range(
    1,
    NUMBER_OF_RUNS + 1
):

    print(
        f"\nStarting MaxViT resource run "
        f"{run_number}/{NUMBER_OF_RUNS}"
    )

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    time.sleep(COOLDOWN_SECONDS)

    monitor = ResourceMonitor(
        interval=SAMPLING_INTERVAL,
        gpu_index=GPU_INDEX
    )

    monitor.start()

    # Ensure no previous CUDA operations remain queued
    if device.type == "cuda":
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    processed_images = 0
    last_logits = None

    with torch.inference_mode():

        for images, labels in test_loader:

            images = images.to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

            # timm MaxViT forward pass
            logits = model(images)

            last_logits = logits

            processed_images += images.size(0)


    # Wait for all CUDA inference operations
    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed_time = (
        time.perf_counter()
        - start_time
    )


    if last_logits is None:
        monitor.stop(
            elapsed_seconds=0.0,
            number_of_images=0
        )

        raise RuntimeError(
            "No images were processed during inference."
        )


    last_output_value = float(
        last_logits[-1, 0]
        .detach()
        .cpu()
        .item()
    )


    run_summary = monitor.stop(
        elapsed_seconds=elapsed_time,
        number_of_images=processed_images
    )

    run_summary["run"] = run_number

    run_summary[
        "processed_images"
    ] = processed_images

    run_summary[
        "last_output_value"
    ] = last_output_value

    run_results.append(
        run_summary
    )


    print(
        f"Run {run_number}: "
        f"{elapsed_time:.2f} seconds | "
        f"{run_summary['latency_ms_per_image']:.4f} "
        f"ms/image | "
        f"{run_summary['throughput_images_per_s']:.2f} "
        f"images/s"
    )


    del last_logits
    del logits
    del images


# ============================================================
# DISPLAY INDIVIDUAL RUNS
# ============================================================

results_df = pd.DataFrame(
    run_results
)

print(
    "\nIndividual MaxViT profiling runs:"
)

display(results_df)

Device: cuda
Test images: 3000
Test batches: 188

Performing warm-up using 3 batches...


C:\Users\aneek\AppData\Local\Temp\ipykernel_41544\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(


Warm-up completed.

Starting MaxViT resource run 1/5
Run 1: 30.06 seconds | 10.0216 ms/image | 99.78 images/s

Starting MaxViT resource run 2/5


C:\Users\aneek\AppData\Local\Temp\ipykernel_41544\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(


Run 2: 29.89 seconds | 9.9644 ms/image | 100.36 images/s

Starting MaxViT resource run 3/5


...

C:\Users\aneek\AppData\Local\Temp\ipykernel_41544\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(


Run 4: 30.15 seconds | 10.0508 ms/image | 99.49 images/s

Starting MaxViT resource run 5/5


C:\Users\aneek\AppData\Local\Temp\ipykernel_41544\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(


Run 5: 29.86 seconds | 9.9543 ms/image | 100.46 images/s

Individual MaxViT profiling runs:


,elapsed_time_s,latency_ms_per_image,throughput_images_per_s,average_cpu_percent,peak_cpu_percent,baseline_ram_mb,average_ram_mb,peak_ram_mb,average_incremental_ram_mb,peak_incremental_ram_mb,...,average_incremental_gpu_memory_mb,peak_incremental_gpu_memory_mb,average_gpu_utilization_percent,peak_gpu_utilization_percent,average_gpu_power_w,peak_gpu_power_w,gpu_energy_wh,run,processed_images,last_output_value
0,30.064795,10.021598,99.784483,3.069580,7.990625,7085.617188,7067.416591,7086.714844,0.000000,1.097656,...,650.421818,682.0,97.996364,100,51.582905,64.055,0.432843,1,3000,-4.575562
1,29.893116,9.964372,100.357552,3.015064,8.434375,7048.136719,7030.146806,7050.605469,0.000000,2.468750,...,1.978022,6.0,98.571429,100,51.964535,59.359,0.433536,2,3000,-4.575562
2,29.998427,9.999476,100.005242,3.150561,7.615625,7002.152344,6972.985462,7006.015625,0.000000,3.863281,...,1.978022,6.0,98.652015,100,51.379993,59.252,0.430250,3,3000,-4.575562
3,30.152278,10.050759,99.494971,3.186170,9.856250,6917.773438,6918.053608,6921.714844,0.280170,3.941406,...,1.978182,6.0,98.570909,100,51.073396,59.295,0.429866,4,3000,-4.575562
4,29.862905,9.954302,100.459082,3.119083,8.512500,6917.863281,6918.054903,6922.683594,0.191622,4.820312,...,1.985294,6.0,98.474265,100,52.041798,59.999,0.433714,5,3000,-4.575562


In [19]:
# ============================================================RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# ============================================================
import numpy as np
import pandas as pd
from scipy.stats import t
metrics_to_report = [
    "elapsed_time_s",
    "latency_ms_per_image",
    "throughput_images_per_s",

    "average_cpu_percent",
    "peak_cpu_percent",

    "average_ram_mb",
    "peak_ram_mb",
    "average_incremental_ram_mb",
    "peak_incremental_ram_mb",

    "average_gpu_memory_mb",
    "peak_gpu_memory_mb",
    "average_incremental_gpu_memory_mb",
    "peak_incremental_gpu_memory_mb",

    "average_gpu_utilization_percent",
    "peak_gpu_utilization_percent",

    "average_gpu_power_w",
    "peak_gpu_power_w",
    "gpu_energy_wh"
]


summary_rows = []

number_of_runs = len(results_df)

for metric in metrics_to_report:
    values = results_df[metric].dropna()

    mean_value = values.mean()
    standard_deviation = values.std(ddof=1)

    if len(values) > 1:
        critical_t = t.ppf(
            0.975,
            df=len(values) - 1
        )

        confidence_half_width = (
            critical_t
            * standard_deviation
            / np.sqrt(len(values))
        )
    else:
        confidence_half_width = np.nan

    summary_rows.append({
        "Metric": metric,
        "Mean": mean_value,
        "Standard Deviation": standard_deviation,
        "95% CI Lower": (
            mean_value - confidence_half_width
        ),
        "95% CI Upper": (
            mean_value + confidence_half_width
        )
    })


resource_summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 90)
print("MaXVit RESOURCE CONSUMPTION — FIVE INFERENCE RUNS")
print("=" * 90)

display(resource_summary)


print("\nMain values for the manuscript")
print("-" * 90)

for metric in [
    "latency_ms_per_image",
    "peak_ram_mb",
    "peak_gpu_memory_mb",
    "average_gpu_utilization_percent",
    "average_gpu_power_w"
]:
    row = resource_summary[
        resource_summary["Metric"] == metric
    ].iloc[0]

    print(
        f"{metric}: "
        f"{row['Mean']:.3f} ± "
        f"{row['Standard Deviation']:.3f} "
        f"(95% CI: "
        f"{row['95% CI Lower']:.3f}–"
        f"{row['95% CI Upper']:.3f})"
    )


MaXVit RESOURCE CONSUMPTION — FIVE INFERENCE RUNS


,Metric,Mean,Standard Deviation,95% CI Lower,95% CI Upper
0,elapsed_time_s,29.994304,0.119840,29.845503,30.143105
1,latency_ms_per_image,9.998101,0.039947,9.948501,10.047702
2,throughput_images_per_s,100.020266,0.399398,99.524348,100.516184
3,average_cpu_percent,3.108092,0.067376,3.024433,3.191750
4,peak_cpu_percent,8.481875,0.849090,7.427590,9.536160
5,average_ram_mb,6981.331474,66.841702,6898.336539,7064.326410
6,peak_ram_mb,6997.546875,74.486355,6905.059849,7090.033901
7,average_incremental_ram_mb,0.094358,0.132944,-0.070714,0.259431
8,peak_incremental_ram_mb,3.238281,1.463069,1.421641,5.054922
9,average_gpu_memory_mb,7333.038580,12.324340,7317.735890,7348.341270



Main values for the manuscript
------------------------------------------------------------------------------------------
latency_ms_per_image: 9.998 ± 0.040 (95% CI: 9.949–10.048)
peak_ram_mb: 6997.547 ± 74.486 (95% CI: 6905.060–7090.034)
peak_gpu_memory_mb: 7342.570 ± 0.000 (95% CI: 7342.570–7342.570)
average_gpu_utilization_percent: 98.453 ± 0.263 (95% CI: 98.127–98.779)
average_gpu_power_w: 51.609 ± 0.404 (95% CI: 51.107–52.110)


#genralization

In [21]:
#wild deepfake on dfc
print("\nTest results of DFC on wild deepfake dataset (maxvit):")
test_dataset = DeepfakeMaxViTDataset(images=test_images,labels=test_labels,transform=maxvit_transform)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results = evaluate_complete(model=model,loader=test_loader,criterion=criterion,device=device)
print("\nMAXVIT-BASE TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if metric in [
        "confusion_matrix",
        "classification_report",
        "predictions"
    ]:
        continue

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )
    else:
        print(
            f"{metric:30s}: {value}"
        )


Test results of DFC on wild deepfake dataset (maxvit):


C:\Users\aneek\AppData\Local\Temp\ipykernel_41544\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(



MAXVIT-BASE TEST RESULTS
test_loss                     : 1.703712
accuracy                      : 0.746889
balanced_accuracy             : 0.505556
precision                     : 0.752114
recall_sensitivity            : 0.988222
specificity                   : 0.022889
f1_score                      : 0.854152
mcc                           : 0.040172
roc_auc                       : 0.605117
pr_auc                        : 0.808526
average_precision             : 0.808550
eer                           : 0.415407
eer_threshold                 : 0.999546
false_positive_rate           : 0.977111
false_negative_rate           : 0.011778
true_negatives                : 103
false_positives               : 4397
false_negatives               : 159
true_positives                : 13341
number_of_test_images         : 18000


In [23]:
#celeb on dfc
print("\nTest results of DFC on Celeb-DF(V2) dataset (maxvit):")
test_dataset = DeepfakeMaxViTDataset(images=test_celeb,labels=test_labels,transform=maxvit_transform)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results = evaluate_complete(model=model,loader=test_loader,criterion=criterion,device=device)
print("\nMAXVIT-BASE TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if metric in [
        "confusion_matrix",
        "classification_report",
        "predictions"
    ]:
        continue

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )
    else:
        print(
            f"{metric:30s}: {value}"
        )


Test results of DFC on Celeb-DF(V2) dataset (maxvit):


C:\Users\aneek\AppData\Local\Temp\ipykernel_41544\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(



MAXVIT-BASE TEST RESULTS
test_loss                     : 0.801475
accuracy                      : 0.893978
balanced_accuracy             : 0.503736
precision                     : 0.904867
recall_sensitivity            : 0.986456
specificity                   : 0.021016
f1_score                      : 0.943902
mcc                           : 0.018548
roc_auc                       : 0.517189
pr_auc                        : 0.907992
average_precision             : 0.908014
eer                           : 0.483518
eer_threshold                 : 0.999838
false_positive_rate           : 0.978984
false_negative_rate           : 0.013544
true_negatives                : 12
false_positives               : 559
false_negatives               : 73
true_positives                : 5317
number_of_test_images         : 5961


In [25]:
#FF++ on hog
print("\nTest results of dfc on FF++ dataset (maxvit):")
test_dataset = DeepfakeMaxViTDataset(images=test_ff,labels=test_ff_labels,transform=maxvit_transform)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results = evaluate_complete(model=model,loader=test_loader,criterion=criterion,device=device)
print("\nMAXVIT-BASE TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if metric in [
        "confusion_matrix",
        "classification_report",
        "predictions"
    ]:
        continue

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )
    else:
        print(
            f"{metric:30s}: {value}"
        )


Test results of dfc on FF++ dataset (maxvit):


C:\Users\aneek\AppData\Local\Temp\ipykernel_41544\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(



MAXVIT-BASE TEST RESULTS
test_loss                     : 4.038546
accuracy                      : 0.446465
balanced_accuracy             : 0.516885
precision                     : 0.425495
recall_sensitivity            : 0.938894
specificity                   : 0.094875
f1_score                      : 0.585602
mcc                           : 0.061085
roc_auc                       : 0.549190
pr_auc                        : 0.452622
average_precision             : 0.453282
eer                           : 0.460276
eer_threshold                 : 0.999667
false_positive_rate           : 0.905125
false_negative_rate           : 0.061106
true_negatives                : 137
false_positives               : 1307
false_negatives               : 63
true_positives                : 968
number_of_test_images         : 2475


# FF++

LOAD THE DATASET

In [24]:
import os, cv2, numpy as np

FINAL_ROOT = r'D:\thesis\ff_final'

def load_split(split, cls):
    base = os.path.join(FINAL_ROOT, split, cls)
    nested, ids = [], []
    for vid_id in sorted(os.listdir(base)):
        d = os.path.join(base, vid_id)
        frames = [cv2.imread(os.path.join(d, f)) for f in sorted(os.listdir(d))]
        if frames:
            nested.append(frames); ids.append(vid_id)
    return nested, ids

# Main splits
ff_real_train_f, ff_real_train_ids = load_split('train', 'real')
ff_fake_train_f, ff_fake_train_ids = load_split('train', 'fake')
ff_real_val_f,   ff_real_val_ids   = load_split('val',   'real')
ff_fake_val_f,   ff_fake_val_ids   = load_split('val',   'fake')
ff_real_test_f,  ff_real_test_ids  = load_split('test',  'real')
ff_fake_test_f,  ff_fake_test_ids  = load_split('test',  'fake')

print("Reloaded main splits. Example shape:", np.shape(ff_real_train_f[0][0]))  # (160,160,3)
print("Real train videos:", len(ff_real_train_f), "| Fake train videos:", len(ff_fake_train_f))
import numpy as np


def combine_split(real_videos, fake_videos):
    """
    Combine all frames from the real and fake video groups.

    Labels:
        0 = Real
        1 = Fake
    """

    real_frames = [
        frame
        for video in real_videos
        for frame in video
        if frame is not None
    ]

    fake_frames = [
        frame
        for video in fake_videos
        for frame in video
        if frame is not None
    ]

    if not real_frames:
        raise ValueError("No real frames found.")

    if not fake_frames:
        raise ValueError("No fake frames found.")

    real_frames = np.stack(real_frames).astype(np.uint8)
    fake_frames = np.stack(fake_frames).astype(np.uint8)

    images = np.concatenate(
        [real_frames, fake_frames],
        axis=0
    )

    real_labels = np.zeros(
        len(real_frames),
        dtype=np.uint8
    )

    fake_labels = np.ones(
        len(fake_frames),
        dtype=np.uint8
    )

    labels = np.concatenate(
        [real_labels, fake_labels],
        axis=0
    )

    return images, labels
# Training data
train_ff, train_ff_labels = combine_split(
    ff_real_train_f,
    ff_fake_train_f
)

# Validation data
val_ff, val_ff_labels = combine_split(
    ff_real_val_f,
    ff_fake_val_f
)

# Testing data
test_ff, test_ff_labels = combine_split(
    ff_real_test_f,
    ff_fake_test_f
)
print("\nTRAIN")
print("Images:", train_ff.shape)
print("Labels:", train_ff_labels.shape)
print("Real:", np.sum(train_ff_labels == 0))
print("Fake:", np.sum(train_ff_labels == 1))

print("\nVALIDATION")
print("Images:", val_ff.shape)
print("Labels:", val_ff_labels.shape)
print("Real:", np.sum(val_ff_labels == 0))
print("Fake:", np.sum(val_ff_labels == 1))

print("\nTEST")
print("Images:", test_ff.shape)
print("Labels:", test_ff_labels.shape)
print("Real:", np.sum(test_ff_labels == 0))
print("Fake:", np.sum(test_ff_labels == 1))

print("\nData types")
print("Train images:", train_ff.dtype)
print("Train labels:", train_ff_labels.dtype)

Reloaded main splits. Example shape: (160, 160, 3)
Real train videos: 517 | Fake train videos: 320

TRAIN
Images: (4595, 160, 160, 3)
Labels: (4595,)
Real: 2948
Fake: 1647

VALIDATION
Images: (948, 160, 160, 3)
Labels: (948,)
Real: 499
Fake: 449

TEST
Images: (2475, 160, 160, 3)
Labels: (2475,)
Real: 1444
Fake: 1031

Data types
Train images: uint8
Train labels: uint8


In [11]:
train_dataset = DeepfakeMaxViTDataset(images=train_ff,labels=train_ff_labels,transform=maxvit_transform)
val_dataset = DeepfakeMaxViTDataset(images=val_ff,labels=val_ff_labels,transform=maxvit_transform)
test_dataset = DeepfakeMaxViTDataset(images=test_ff,labels=test_ff_labels,transform=maxvit_transform)

train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())


print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Testing samples:", len(test_dataset))

Training samples: 4595
Validation samples: 948
Testing samples: 2475


In [12]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    epochs=10
)

C:\Users\aneek\AppData\Local\Temp\ipykernel_37016\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(


Epoch 01/10 | Train Loss: 0.2850 | Train Accuracy: 0.8736 | Val Loss: 0.1831 | Val Accuracy: 0.9167
Epoch 02/10 | Train Loss: 0.1083 | Train Accuracy: 0.9571 | Val Loss: 0.1309 | Val Accuracy: 0.9451
Epoch 03/10 | Train Loss: 0.0866 | Train Accuracy: 0.9669 | Val Loss: 0.1767 | Val Accuracy: 0.9241
Epoch 04/10 | Train Loss: 0.0557 | Train Accuracy: 0.9785 | Val Loss: 0.1712 | Val Accuracy: 0.9314
Epoch 05/10 | Train Loss: 0.0395 | Train Accuracy: 0.9850 | Val Loss: 0.5552 | Val Accuracy: 0.8259
Epoch 06/10 | Train Loss: 0.0457 | Train Accuracy: 0.9843 | Val Loss: 0.1101 | Val Accuracy: 0.9504
Epoch 07/10 | Train Loss: 0.0208 | Train Accuracy: 0.9939 | Val Loss: 0.1138 | Val Accuracy: 0.9652
Epoch 08/10 | Train Loss: 0.0277 | Train Accuracy: 0.9915 | Val Loss: 0.1049 | Val Accuracy: 0.9589
Epoch 09/10 | Train Loss: 0.0250 | Train Accuracy: 0.9919 | Val Loss: 0.1461 | Val Accuracy: 0.9430
Epoch 10/10 | Train Loss: 0.0224 | Train Accuracy: 0.9924 | Val Loss: 0.2310 | Val Accuracy: 0.9325


In [13]:
print("=== DATA LOADING ===")
start = monitor.get_stats()

=== DATA LOADING ===


In [14]:
print("=== DATA LOADING ===")
start = monitor1.get_stats()

=== DATA LOADING ===


In [15]:
test_results = evaluate_complete(
    model=model,
    loader=test_loader,
    criterion=criterion,
    device=device
)
print("\nMAXVIT-BASE TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if metric in [
        "confusion_matrix",
        "classification_report",
        "predictions"
    ]:
        continue

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )
    else:
        print(
            f"{metric:30s}: {value}"
        )

C:\Users\aneek\AppData\Local\Temp\ipykernel_37016\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(



MAXVIT-BASE TEST RESULTS
test_loss                     : 0.166755
accuracy                      : 0.949899
balanced_accuracy             : 0.946106
precision                     : 0.954865
recall_sensitivity            : 0.923375
specificity                   : 0.968837
f1_score                      : 0.938856
mcc                           : 0.896801
roc_auc                       : 0.989867
pr_auc                        : 0.987208
average_precision             : 0.987214
eer                           : 0.052850
eer_threshold                 : 0.174216
false_positive_rate           : 0.031163
false_negative_rate           : 0.076625
true_negatives                : 1399
false_positives               : 45
false_negatives               : 79
true_positives                : 952
number_of_test_images         : 2475


In [16]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 11.5%
Time Usage: 33.6 s
GPU Memory Used: 0.0 MB
Power Consumption: 93W


In [17]:
# Your data loading operations here
time.sleep(2)  # Simulate loading time

end = monitor1.get_stats()
duration = end['timestamp'] - start['timestamp']

print("\n=== RESOURCE USAGE ===")
print(f"CPU Usage: {end['cpu_%']:.1f}%")
print(f"Time Usage: {duration:.1f} s")
print(f"GPU Memory Used: {end['gpu_mem_mb']:.1f} MB")
print(f"Power Consumption: {int(end['power_w'])}W")  # Rounded to whole watts


=== RESOURCE USAGE ===
CPU Usage: 12.1%
Time Usage: 35.8 s
GPU Memory Used: 1845.1 MB
Power Consumption: 93W


#save the model

In [18]:
# ============================================================
# SAVE TIMM MAXVIT MODEL
# ============================================================

import os
import torch


SAVE_DIR = os.path.join(
    r"D:\thesis\results",
    "maxvit_ff_160"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)


CHECKPOINT_PATH = os.path.join(
    SAVE_DIR,
    "training_checkpoint.pt"
)

WEIGHTS_PATH = os.path.join(
    SAVE_DIR,
    "maxvit_weights.pth"
)


# Save model weights only
torch.save(
    model.state_dict(),
    WEIGHTS_PATH
)


# Save full training checkpoint
torch.save(
    {
        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "history":
            history,

        "model_name":
            MODEL_NAME,

        "input_size":
            INPUT_SIZE,

        "num_classes":
            NUM_CLASSES,

        "epochs":
            EPOCHS,

        "batch_size":
            BATCH_SIZE,

        "learning_rate":
            LEARNING_RATE,

        "random_seed":
            RANDOM_SEED,

        "label_mapping": {
            0: "real",
            1: "fake"
        },

        "normalization_mean": [
            0.485,
            0.456,
            0.406
        ],

        "normalization_std": [
            0.229,
            0.224,
            0.225
        ]
    },

    CHECKPOINT_PATH
)


print("MaxViT model saved successfully.")
print("Weights:", WEIGHTS_PATH)
print("Checkpoint:", CHECKPOINT_PATH)

MaxViT model saved successfully.
Weights: D:\thesis\results\maxvit_ff_160\maxvit_weights.pth
Checkpoint: D:\thesis\results\maxvit_ff_160\training_checkpoint.pt


In [19]:
# ============================================================
# RESOURCE MONITOR DEFINITION
# RUN THIS BEFORE THE ViT PROFILING CELL
# ============================================================

import os
import time
import threading
import numpy as np
import pandas as pd
import psutil

from pynvml import (
    nvmlInit,
    nvmlShutdown,
    nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetMemoryInfo,
    nvmlDeviceGetUtilizationRates,
    nvmlDeviceGetPowerUsage,
    NVMLError
)


SAMPLING_INTERVAL = 0.1
GPU_INDEX = 0


class ResourceMonitor:
    """
    Continuously samples CPU, RAM, GPU memory,
    GPU utilization, and GPU power.
    """

    def __init__(
        self,
        interval=0.1,
        gpu_index=0
    ):
        self.interval = interval

        self.process = psutil.Process(
            os.getpid()
        )

        self.logical_cpu_count = (
            psutil.cpu_count(logical=True) or 1
        )

        self.stop_event = threading.Event()
        self.samples = []
        self.thread = None

        nvmlInit()

        self.gpu_handle = (
            nvmlDeviceGetHandleByIndex(
                gpu_index
            )
        )

    def _read_gpu_power(self):
        try:
            return (
                nvmlDeviceGetPowerUsage(
                    self.gpu_handle
                ) / 1000.0
            )
        except NVMLError:
            return np.nan

    def _collect_sample(self):
        timestamp = time.perf_counter()

        ram_mb = (
            self.process.memory_info().rss
            / (1024 ** 2)
        )

        process_cpu_raw = (
            self.process.cpu_percent(
                interval=None
            )
        )

        process_cpu_normalized = (
            process_cpu_raw
            / self.logical_cpu_count
        )

        gpu_memory = nvmlDeviceGetMemoryInfo(
            self.gpu_handle
        )

        gpu_memory_used_mb = (
            gpu_memory.used
            / (1024 ** 2)
        )

        gpu_utilization = (
            nvmlDeviceGetUtilizationRates(
                self.gpu_handle
            ).gpu
        )

        gpu_power_w = self._read_gpu_power()

        self.samples.append({
            "timestamp": timestamp,
            "ram_mb": ram_mb,
            "cpu_percent": process_cpu_normalized,
            "gpu_memory_mb": gpu_memory_used_mb,
            "gpu_utilization_percent": gpu_utilization,
            "gpu_power_w": gpu_power_w
        })

    def _sampling_loop(self):
        while not self.stop_event.is_set():
            try:
                self._collect_sample()
            except Exception as error:
                print(
                    "Monitoring warning:",
                    error
                )

            self.stop_event.wait(
                self.interval
            )

    def start(self):
        # Initialize the CPU utilization counter
        self.process.cpu_percent(
            interval=None
        )

        # First observation is the baseline
        self._collect_sample()

        self.thread = threading.Thread(
            target=self._sampling_loop,
            daemon=True
        )

        self.thread.start()

    def stop(
        self,
        elapsed_seconds,
        number_of_images
    ):
        self.stop_event.set()

        if self.thread is not None:
            self.thread.join()

        try:
            self._collect_sample()
        except Exception:
            pass

        data = pd.DataFrame(
            self.samples
        )

        nvmlShutdown()

        if data.empty:
            raise RuntimeError(
                "No resource samples were collected."
            )

        baseline_ram = data[
            "ram_mb"
        ].iloc[0]

        baseline_gpu_memory = data[
            "gpu_memory_mb"
        ].iloc[0]

        average_ram = data[
            "ram_mb"
        ].mean()

        peak_ram = data[
            "ram_mb"
        ].max()

        average_gpu_memory = data[
            "gpu_memory_mb"
        ].mean()

        peak_gpu_memory = data[
            "gpu_memory_mb"
        ].max()

        average_incremental_ram = max(
            0.0,
            average_ram - baseline_ram
        )

        peak_incremental_ram = max(
            0.0,
            peak_ram - baseline_ram
        )

        average_incremental_gpu_memory = max(
            0.0,
            average_gpu_memory
            - baseline_gpu_memory
        )

        peak_incremental_gpu_memory = max(
            0.0,
            peak_gpu_memory
            - baseline_gpu_memory
        )

        # Integrate GPU power over time
        valid_power = data.dropna(
            subset=["gpu_power_w"]
        )

        if len(valid_power) >= 2:
            relative_times = (
                valid_power[
                    "timestamp"
                ].to_numpy()
                - valid_power[
                    "timestamp"
                ].iloc[0]
            )

            # Compatibility with different NumPy versions
            if hasattr(np, "trapezoid"):
                energy_joules = np.trapezoid(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )
            else:
                energy_joules = np.trapz(
                    valid_power[
                        "gpu_power_w"
                    ].to_numpy(),
                    relative_times
                )

            energy_wh = (
                energy_joules / 3600.0
            )
        else:
            energy_wh = np.nan

        return {
            "elapsed_time_s":
                elapsed_seconds,

            "latency_ms_per_image":
                elapsed_seconds
                / number_of_images
                * 1000.0,

            "throughput_images_per_s":
                number_of_images
                / elapsed_seconds,

            "average_cpu_percent":
                data[
                    "cpu_percent"
                ].mean(),

            "peak_cpu_percent":
                data[
                    "cpu_percent"
                ].max(),

            "baseline_ram_mb":
                baseline_ram,

            "average_ram_mb":
                average_ram,

            "peak_ram_mb":
                peak_ram,

            "average_incremental_ram_mb":
                average_incremental_ram,

            "peak_incremental_ram_mb":
                peak_incremental_ram,

            "baseline_gpu_memory_mb":
                baseline_gpu_memory,

            "average_gpu_memory_mb":
                average_gpu_memory,

            "peak_gpu_memory_mb":
                peak_gpu_memory,

            "average_incremental_gpu_memory_mb":
                average_incremental_gpu_memory,

            "peak_incremental_gpu_memory_mb":
                peak_incremental_gpu_memory,

            "average_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].mean(),

            "peak_gpu_utilization_percent":
                data[
                    "gpu_utilization_percent"
                ].max(),

            "average_gpu_power_w":
                data[
                    "gpu_power_w"
                ].mean(),

            "peak_gpu_power_w":
                data[
                    "gpu_power_w"
                ].max(),

            "gpu_energy_wh":
                energy_wh
        }


print("ResourceMonitor defined successfully.")

ResourceMonitor defined successfully.


C:\Users\aneek\AppData\Local\Temp\ipykernel_37016\2400127911.py:13: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  from pynvml import (


In [20]:
# ============================================================
# PYTORCH MAXVIT WARM-UP AND REPEATED INFERENCE PROFILING
# ============================================================

import gc
import time
import torch
import pandas as pd


NUMBER_OF_RUNS = 5
WARMUP_BATCHES = 3
COOLDOWN_SECONDS = 5


# ============================================================
# DEVICE AND MODEL
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = model.to(device)
model.eval()

number_of_test_images = len(
    test_loader.dataset
)

print("Device:", device)
print("Test images:", number_of_test_images)
print("Test batches:", len(test_loader))


# ============================================================
# GPU/MODEL WARM-UP
# ============================================================

print(
    f"\nPerforming warm-up using "
    f"{WARMUP_BATCHES} batches..."
)

with torch.inference_mode():

    for batch_index, batch in enumerate(
        test_loader
    ):

        if batch_index >= WARMUP_BATCHES:
            break

        # MaxViT dataset returns:
        # images tensor, labels tensor
        images, labels = batch

        images = images.to(
            device,
            dtype=torch.float32,
            non_blocking=True
        )

        # timm MaxViT returns logits directly
        logits = model(images)

        # Materialize part of the output
        _ = logits[-1, 0].item()


if device.type == "cuda":
    torch.cuda.synchronize()

print("Warm-up completed.")


# ============================================================
# REPEATED INFERENCE RUNS
# ============================================================

run_results = []

for run_number in range(
    1,
    NUMBER_OF_RUNS + 1
):

    print(
        f"\nStarting MaxViT resource run "
        f"{run_number}/{NUMBER_OF_RUNS}"
    )

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    time.sleep(COOLDOWN_SECONDS)

    monitor = ResourceMonitor(
        interval=SAMPLING_INTERVAL,
        gpu_index=GPU_INDEX
    )

    monitor.start()

    # Ensure no previous CUDA operations remain queued
    if device.type == "cuda":
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    processed_images = 0
    last_logits = None

    with torch.inference_mode():

        for images, labels in test_loader:

            images = images.to(
                device,
                dtype=torch.float32,
                non_blocking=True
            )

            # timm MaxViT forward pass
            logits = model(images)

            last_logits = logits

            processed_images += images.size(0)


    # Wait for all CUDA inference operations
    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed_time = (
        time.perf_counter()
        - start_time
    )


    if last_logits is None:
        monitor.stop(
            elapsed_seconds=0.0,
            number_of_images=0
        )

        raise RuntimeError(
            "No images were processed during inference."
        )


    last_output_value = float(
        last_logits[-1, 0]
        .detach()
        .cpu()
        .item()
    )


    run_summary = monitor.stop(
        elapsed_seconds=elapsed_time,
        number_of_images=processed_images
    )

    run_summary["run"] = run_number

    run_summary[
        "processed_images"
    ] = processed_images

    run_summary[
        "last_output_value"
    ] = last_output_value

    run_results.append(
        run_summary
    )


    print(
        f"Run {run_number}: "
        f"{elapsed_time:.2f} seconds | "
        f"{run_summary['latency_ms_per_image']:.4f} "
        f"ms/image | "
        f"{run_summary['throughput_images_per_s']:.2f} "
        f"images/s"
    )


    del last_logits
    del logits
    del images


# ============================================================
# DISPLAY INDIVIDUAL RUNS
# ============================================================

results_df = pd.DataFrame(
    run_results
)

print(
    "\nIndividual MaxViT profiling runs:"
)

display(results_df)

Device: cuda
Test images: 2475
Test batches: 155

Performing warm-up using 3 batches...


C:\Users\aneek\AppData\Local\Temp\ipykernel_37016\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(


Warm-up completed.

Starting MaxViT resource run 1/5
Run 1: 25.28 seconds | 10.2147 ms/image | 97.90 images/s

Starting MaxViT resource run 2/5


C:\Users\aneek\AppData\Local\Temp\ipykernel_37016\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(


Run 2: 24.77 seconds | 10.0079 ms/image | 99.92 images/s

Starting MaxViT resource run 3/5


...

C:\Users\aneek\AppData\Local\Temp\ipykernel_37016\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(


Run 4: 24.98 seconds | 10.0923 ms/image | 99.09 images/s

Starting MaxViT resource run 5/5


C:\Users\aneek\AppData\Local\Temp\ipykernel_37016\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(


Run 5: 25.12 seconds | 10.1499 ms/image | 98.52 images/s

Individual MaxViT profiling runs:


,elapsed_time_s,latency_ms_per_image,throughput_images_per_s,average_cpu_percent,peak_cpu_percent,baseline_ram_mb,average_ram_mb,peak_ram_mb,average_incremental_ram_mb,peak_incremental_ram_mb,...,average_incremental_gpu_memory_mb,peak_incremental_gpu_memory_mb,average_gpu_utilization_percent,peak_gpu_utilization_percent,average_gpu_power_w,peak_gpu_power_w,gpu_energy_wh,run,processed_images,last_output_value
0,25.281345,10.214685,97.898273,2.752061,7.168750,6441.847656,6426.735385,6442.050781,0.000000,0.203125,...,593.422414,774.0,97.435345,100,50.376634,68.985,0.356069,1,2475,-1.972031
1,24.769552,10.007900,99.921062,2.993225,8.878125,6410.328125,6388.827382,6414.328125,0.000000,4.000000,...,1.982301,8.0,98.238938,100,51.636305,58.575,0.357291,2,2475,-1.972031
2,24.716588,9.986500,100.135180,3.143612,7.546875,6354.382812,6173.557021,6354.507812,0.000000,0.125000,...,1.991150,8.0,98.548673,100,51.812695,61.267,0.357761,3,2475,-1.972031
3,24.978380,10.092275,99.085691,2.910526,8.434375,5907.910156,5908.094658,5912.726562,0.184502,4.816406,...,1.982456,8.0,98.385965,100,50.765724,58.427,0.354396,4,2475,-1.972031
4,25.120902,10.149860,98.523531,3.040038,8.878125,5907.933594,5908.430489,5913.035156,0.496895,5.101562,...,1.982533,8.0,98.296943,100,50.276568,56.776,0.352792,5,2475,-1.972031


In [21]:
# ============================================================RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# RESOURCE RESULTS: MEAN, SD AND 95% CONFIDENCE INTERVAL
# ============================================================
import numpy as np
import pandas as pd
from scipy.stats import t
metrics_to_report = [
    "elapsed_time_s",
    "latency_ms_per_image",
    "throughput_images_per_s",

    "average_cpu_percent",
    "peak_cpu_percent",

    "average_ram_mb",
    "peak_ram_mb",
    "average_incremental_ram_mb",
    "peak_incremental_ram_mb",

    "average_gpu_memory_mb",
    "peak_gpu_memory_mb",
    "average_incremental_gpu_memory_mb",
    "peak_incremental_gpu_memory_mb",

    "average_gpu_utilization_percent",
    "peak_gpu_utilization_percent",

    "average_gpu_power_w",
    "peak_gpu_power_w",
    "gpu_energy_wh"
]


summary_rows = []

number_of_runs = len(results_df)

for metric in metrics_to_report:
    values = results_df[metric].dropna()

    mean_value = values.mean()
    standard_deviation = values.std(ddof=1)

    if len(values) > 1:
        critical_t = t.ppf(
            0.975,
            df=len(values) - 1
        )

        confidence_half_width = (
            critical_t
            * standard_deviation
            / np.sqrt(len(values))
        )
    else:
        confidence_half_width = np.nan

    summary_rows.append({
        "Metric": metric,
        "Mean": mean_value,
        "Standard Deviation": standard_deviation,
        "95% CI Lower": (
            mean_value - confidence_half_width
        ),
        "95% CI Upper": (
            mean_value + confidence_half_width
        )
    })


resource_summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 90)
print("MaXViT RESOURCE CONSUMPTION — FIVE INFERENCE RUNS")
print("=" * 90)

display(resource_summary)


print("\nMain values for the manuscript")
print("-" * 90)

for metric in [
    "latency_ms_per_image",
    "peak_ram_mb",
    "peak_gpu_memory_mb",
    "average_gpu_utilization_percent",
    "average_gpu_power_w"
]:
    row = resource_summary[
        resource_summary["Metric"] == metric
    ].iloc[0]

    print(
        f"{metric}: "
        f"{row['Mean']:.3f} ± "
        f"{row['Standard Deviation']:.3f} "
        f"(95% CI: "
        f"{row['95% CI Lower']:.3f}–"
        f"{row['95% CI Upper']:.3f})"
    )


MaXViT RESOURCE CONSUMPTION — FIVE INFERENCE RUNS


,Metric,Mean,Standard Deviation,95% CI Lower,95% CI Upper
0,elapsed_time_s,24.973353,0.236705,24.679445,25.267262
1,latency_ms_per_image,10.090244,0.095639,9.971493,10.208995
2,throughput_images_per_s,99.112748,0.938296,97.947699,100.277796
3,average_cpu_percent,2.967892,0.147138,2.785197,3.150588
4,peak_cpu_percent,8.181250,0.784686,7.206933,9.155567
5,average_ram_mb,6161.128987,250.215245,5850.445725,6471.812249
6,peak_ram_mb,6207.329687,270.649228,5871.274285,6543.385090
7,average_incremental_ram_mb,0.136279,0.216844,-0.132968,0.405527
8,peak_incremental_ram_mb,2.849219,2.484470,-0.235658,5.934095
9,average_gpu_memory_mb,7301.329983,78.066588,7204.397512,7398.262454



Main values for the manuscript
------------------------------------------------------------------------------------------
latency_ms_per_image: 10.090 ± 0.096 (95% CI: 9.971–10.209)
peak_ram_mb: 6207.330 ± 270.649 (95% CI: 5871.274–6543.385)
peak_gpu_memory_mb: 7342.258 ± 0.000 (95% CI: 7342.258–7342.258)
average_gpu_utilization_percent: 98.181 ± 0.433 (95% CI: 97.644–98.719)
average_gpu_power_w: 50.974 ± 0.712 (95% CI: 50.089–51.858)


#gernalization

In [23]:
#wild deepfake on ff
print("\nTest results of FF++ on wild deepfake dataset SwinV2:")
test_dataset = DeepfakeMaxViTDataset(images=test_images,labels=test_labels,transform=maxvit_transform)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results = evaluate_complete(model=model,loader=test_loader,criterion=criterion,device=device)
print("\nMAXVIT-BASE TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if metric in [
        "confusion_matrix",
        "classification_report",
        "predictions"
    ]:
        continue

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )
    else:
        print(
            f"{metric:30s}: {value}"
        )


Test results of FF++ on wild deepfake dataset SwinV2:


C:\Users\aneek\AppData\Local\Temp\ipykernel_37016\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(



MAXVIT-BASE TEST RESULTS
test_loss                     : 2.900516
accuracy                      : 0.353000
balanced_accuracy             : 0.516593
precision                     : 0.784356
recall_sensitivity            : 0.189407
specificity                   : 0.843778
f1_score                      : 0.305131
mcc                           : 0.037313
roc_auc                       : 0.488555
pr_auc                        : 0.760187
average_precision             : 0.760223
eer                           : 0.516741
eer_threshold                 : 0.034916
false_positive_rate           : 0.156222
false_negative_rate           : 0.810593
true_negatives                : 3797
false_positives               : 703
false_negatives               : 10943
true_positives                : 2557
number_of_test_images         : 18000


In [25]:
#celeb on ff
print("\nTest results of FF++ on Celeb-df(v2) dataset (maxvit):")
test_dataset = DeepfakeMaxViTDataset(images=test_celeb,labels=test_labels,transform=maxvit_transform)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results = evaluate_complete(model=model,loader=test_loader,criterion=criterion,device=device)
print("\nMAXVIT-BASE TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if metric in [
        "confusion_matrix",
        "classification_report",
        "predictions"
    ]:
        continue

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )
    else:
        print(
            f"{metric:30s}: {value}"
        )


Test results of FF++ on Celeb-df(v2) dataset (maxvit):


C:\Users\aneek\AppData\Local\Temp\ipykernel_37016\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(



MAXVIT-BASE TEST RESULTS
test_loss                     : 5.106443
accuracy                      : 0.199463
balanced_accuracy             : 0.557328
precision                     : 1.000000
recall_sensitivity            : 0.114657
specificity                   : 1.000000
f1_score                      : 0.205726
mcc                           : 0.110694
roc_auc                       : 0.767359
pr_auc                        : 0.971462
average_precision             : 0.971466
eer                           : 0.277964
eer_threshold                 : 0.000270
false_positive_rate           : 0.000000
false_negative_rate           : 0.885343
true_negatives                : 571
false_positives               : 0
false_negatives               : 4772
true_positives                : 618
number_of_test_images         : 5961


In [27]:
#DFC on ff
print("\nTest results of FF++ on DFC dataset (maxvit):")
test_dataset = DeepfakeMaxViTDataset(images=test_hog,labels=test_labels,transform=maxvit_transform)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
test_results = evaluate_complete(model=model,loader=test_loader,criterion=criterion,device=device)
print("\nMAXVIT-BASE TEST RESULTS")
print("=" * 70)

for metric, value in test_results.items():

    if metric in [
        "confusion_matrix",
        "classification_report",
        "predictions"
    ]:
        continue

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{metric:30s}: {value:.6f}"
        )
    else:
        print(
            f"{metric:30s}: {value}"
        )


Test results of FF++ on DFC dataset (maxvit):


C:\Users\aneek\AppData\Local\Temp\ipykernel_37016\510521598.py:68: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(



MAXVIT-BASE TEST RESULTS
test_loss                     : 1.780288
accuracy                      : 0.426333
balanced_accuracy             : 0.426333
precision                     : 0.395656
recall_sensitivity            : 0.279333
specificity                   : 0.573333
f1_score                      : 0.327472
mcc                           : -0.154146
roc_auc                       : 0.393942
pr_auc                        : 0.432554
average_precision             : 0.433611
eer                           : 0.574000
eer_threshold                 : 0.231743
false_positive_rate           : 0.426667
false_negative_rate           : 0.720667
true_negatives                : 860
false_positives               : 640
false_negatives               : 1081
true_positives                : 419
number_of_test_images         : 3000
